# 🎬 Spot Republic · versión CINE (tráiler + humor) con órdenes por voz

**Estructura:** TRÁILER épico (tomas de cine, barras negras, voz de narrador, música de tráiler generada, BRAAAM) → **rayón de disco** → TU HUMOR (subtítulos amarillos, pantallas negras sarcásticas, b-roll, zooms de risa, respuestas del editor) → CIERRE épico → BLOOPERS.
**Look de película:** halación, grano, destellos, vibración de proyector · **Desenfoque IA** (Depth Anything) · **Cámara lenta IA** (RAFT) · **Diseño de sonido** de cine.

## 🎙️ Dirige al editor con tu voz mientras grabas
Di **"Editor,"** + la orden, con una pausa antes y después. La orden se aplica a lo que sigue y **se corta del video**. Para que sí se vea, di antes **"Editor, deja esto"**.

| Transiciones (al terminar una toma) | Efectos (sobre lo que sigue) |
|---|---|
| "Editor, transición **rayón**" (aquí termina el tráiler) | "Editor, **cámara lenta**" · "Editor, **zoom**" / "**acércate**" |
| "Editor, transición **barrido**" | "Editor, **modo cómic**" · "Editor, **neón**" |
| "Editor, transición **flash**" | "Editor, **blanco y negro**" · "Editor, **temblor**" |
| "Editor, transición **glitch**" | "Editor, **desenfoca**" · "Editor, **tele vieja**" · "Editor, **silencio**" |
| "Editor, transición **zoom**" | "Editor, **pon título**: el gobierno" · "Editor, **pantalla negra**: luego te explico" |
| "Editor, transición **negro**" | "Editor, **muestra** los canales" · "Editor, **pon la de** Colombia" · "Editor, **bloopers**" |

**Carpetas en `Server/`:** clips donde hablas (raíz) · `cine/` tomas sin hablar · `broll/` grabaciones de pantalla · `musica/` · `fuentes/`.
**Cómo usarlo:** Entorno de ejecución → **H100** → **Ejecutar todo**. Resultado en `Server/Edicion_IA/`.

In [ ]:
#@title 1 · Instalar herramientas (Whisper, YOLO, vLLM) y montar Google Drive
#@markdown Usa **Entorno de ejecución → Cambiar tipo → GPU H100 (o A100 80 GB)**. Tarda ~3-5 min.
import subprocess, sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'faster-whisper', 'ultralytics', 'transformers', 'anthropic>=0.70'], check=True)
    # vLLM va en un entorno aparte para no romper el torch de Colab (usado por CLIP/YOLO/Whisper)
    if not os.path.exists('/content/vllm_env/bin/vllm'):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
        subprocess.run(['uv', 'venv', '-q', '--python', '3.12', '/content/vllm_env'], check=True)
        subprocess.run(['uv', 'pip', 'install', '-q', '--python', '/content/vllm_env/bin/python', '-U', 'vllm'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
try:
    print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'Sin GPU (irá más lento)')
except FileNotFoundError:
    print('Sin GPU (irá más lento)')

In [ ]:
#@title 2 · Configuración — versión CINE (tráiler + humor)
import os, glob, json, math, random, re, shutil, subprocess, base64, gc, time
import numpy as np, cv2

DRIVE_FOLDER   = "/content/drive/MyDrive/Server"  #@param {type:"string"}
NOMBRE_SERVER  = "Spot Republic"  #@param {type:"string"}
#@markdown Qué es el server (la IA lo usa para entender lo que dices y escribir las notas).
DESCRIPCION_SERVER = "Servidor de Discord que funciona como una república: gobierno, presidente, elecciones, partidos políticos (hasta antifurros), ciudadanos, roles y economía, como un país (tipo México). Comunidad con furros."  #@param {type:"string"}
DURACION_OBJETIVO_S = 60  #@param {type:"slider", min:30, max:180, step:5}
#@markdown **Clips:** los videos donde hablas van en `Server/` (como siempre). **B-roll:** grabaciones de pantalla del Discord, capturas, fotos → `Server/broll` (el nombre del archivo ayuda a la IA: `canales.mp4`, `elecciones.png`, `como_entrar.mp4`).
CARPETA_BROLL  = "broll"  #@param {type:"string"}
#@markdown **🎬 Modo cine**: tráiler épico → se rompe con un rayón de disco → tu humor → clímax y "únete".
#@markdown Tomas de cine sin hablar (celular, caminando, azotea, puerta…) → `Server/cine`.
CARPETA_CINE   = "cine"  #@param {type:"string"}
DESENFOQUE_IA  = True  #@param {type:"boolean"}
CAMARA_LENTA_IA = True  #@param {type:"boolean"}
LOOK_PELICULA  = True  #@param {type:"boolean"}
BARRAS_CINE_EN_TRAILER = True  #@param {type:"boolean"}
DISENO_SONIDO_CINE = True  #@param {type:"boolean"}
#@markdown Voz de narrador de tráiler (más grave y con eco) en la parte del tráiler.
VOZ_TRAILER    = True  #@param {type:"boolean"}
#@markdown Música del tráiler: `generada` (épica, sin copyright, hecha por el notebook) o un archivo de `Server/musica`.
MUSICA_TRAILER = "generada"  #@param {type:"string"}
#@markdown **Tono de la IA** para pantallas negras y comentarios.
TONO           = "sarcastico"  #@param ["sarcastico", "serio", "gracioso"]
#@markdown **Tus fuentes**: sube .ttf/.otf a `Server/fuentes` y escribe el nombre del archivo (vacío = la de por defecto).
CARPETA_FUENTES  = "fuentes"  #@param {type:"string"}
FUENTE_SUBTITULOS = ""  #@param {type:"string"}
FUENTE_TARJETAS   = ""  #@param {type:"string"}
FUENTE_TITULOS    = ""  #@param {type:"string"}
#@markdown **Revisión con IA**: antes del video final la IA mira fotogramas de muestra y ajusta el diseño (rondas).
REVISION_IA    = True  #@param {type:"boolean"}
RONDAS_REVISION = 1  #@param {type:"slider", min:1, max:3, step:1}
#@markdown **Groserías**: las tapa con un "piiip" y #$%& en el subtítulo (TikTok les baja el alcance a los videos con muchas).
CENSURAR_GROSERIAS = True  #@param {type:"boolean"}
#@markdown **Efectos**
ZOOMS_RISA     = True  #@param {type:"boolean"}
MUSICA_MOMENTOS_CLAVE = True  #@param {type:"boolean"}
TITULOS_3D_DORADOS = True  #@param {type:"boolean"}
TV_VIEJA       = True  #@param {type:"boolean"}
NEON           = True  #@param {type:"boolean"}
SPIDERVERSE    = True  #@param {type:"boolean"}
#@markdown **Música** (archivos en `Server/musica`). Fondo en orden, separados por coma; vacío = sin música de fondo.
MUSICA_FONDO   = "kwip_flip, caffeine_rush"  #@param {type:"string"}
VOLUMEN_MUSICA = 0.10  #@param {type:"slider", min:0, max:0.5, step:0.01}
#@markdown Canciones a pedido ("editor, pon la de…"): `usar_archivos` las mete desde `Server/musica`; `marcar_para_tiktok` deja el hueco y la lista con el minuto.
MODO_CANCIONES = "usar_archivos"  #@param ["usar_archivos", "marcar_para_tiktok"]
SEGUNDOS_CANCION_PEDIDA = 10  #@param {type:"slider", min:4, max:20, step:1}
VOLUMEN_CANCIONES = 0.8  #@param {type:"slider", min:0.2, max:1, step:0.05}
EFECTOS_SONIDO = True  #@param {type:"boolean"}
CARPETA_MUSICA = "musica"  #@param {type:"string"}
CREDITO_MUSICA = ""  #@param {type:"string"}
#@markdown Texto final (Discord, "link en la bio"…). Vacío = lo propone la IA.
TEXTO_CTA      = ""  #@param {type:"string"}
IDIOMA         = "es"  #@param ["es", "en"]
#@markdown **Motor de IA del editor** · `local_qwen` = Qwen3.8-27B en tu GPU · `claude` = API de Anthropic · `simple` = sin IA.
MOTOR_IA       = "local_qwen"  #@param ["local_qwen", "claude", "simple"]
MODELO_LOCAL   = "Qwen/Qwen3.8-27B"  #@param {type:"string"}
FPS = 30
OUT_W, OUT_H = 1080, 1920
random.seed(7); np.random.seed(7)

if not os.path.isdir(DRIVE_FOLDER):
    raise FileNotFoundError(f"No existe {DRIVE_FOLDER}. Ajusta DRIVE_FOLDER.")
FOLDER = DRIVE_FOLDER
WORK = os.environ.get("SPOT_WORK", "/content/work_v3")
OUT_DIR = os.path.join(FOLDER, "Edicion_IA")
for d in (WORK, f"{WORK}/proxy", f"{WORK}/frames", f"{WORK}/audio", f"{WORK}/broll", OUT_DIR):
    os.makedirs(d, exist_ok=True)
AUDIO_EXT = ('.mp3', '.m4a', '.wav', '.aac', '.ogg', '.flac')
MUSIC_DIR = os.path.join(FOLDER, CARPETA_MUSICA)
SONGS = sorted(f for f in glob.glob(os.path.join(MUSIC_DIR, '*')) if f.lower().endswith(AUDIO_EXT + ('.mp4',)))
CINE_DIR = os.path.join(FOLDER, CARPETA_CINE)
CINE_FILES = sorted(f for f in glob.glob(os.path.join(CINE_DIR, '*')) if f.lower().endswith(('.mp4', '.mov', '.webm', '.mkv')))
BROLL_DIR = os.path.join(FOLDER, CARPETA_BROLL)
BROLL_FILES = sorted(f for f in glob.glob(os.path.join(BROLL_DIR, '*'))
                     if f.lower().endswith(('.mp4', '.mov', '.webm', '.mkv', '.png', '.jpg', '.jpeg', '.webp')))
FONTS_DIR_USER = os.path.join(FOLDER, CARPETA_FUENTES)
USER_FONTS = sorted(f for f in glob.glob(os.path.join(FONTS_DIR_USER, '*')) if f.lower().endswith(('.ttf', '.otf')))
print("Carpeta:", FOLDER, "\nSalida:", OUT_DIR)
print(f"Tus fuentes ({len(USER_FONTS)}):", ', '.join(os.path.basename(f) for f in USER_FONTS) or '—')
print(f"Canciones ({len(SONGS)}):", ', '.join(os.path.basename(f) for f in SONGS) or '—')
print(f"Tomas de cine ({len(CINE_FILES)}):", ', '.join(os.path.basename(f) for f in CINE_FILES) or f"ninguna — súbelas a {CINE_DIR}")
print(f"B-roll ({len(BROLL_FILES)}):", ', '.join(os.path.basename(f) for f in BROLL_FILES) or f"ninguno — súbelo a {BROLL_DIR}")

In [ ]:
#@title 3 · Inventario, quitar duplicados y crear proxies (hasta 1440p para el recorte vertical)
def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"Falló: {' '.join(map(str, cmd))[:300]}\n{r.stderr[-1500:]}")
    return r

def probe(path):
    r = run(['ffprobe', '-v', 'error', '-print_format', 'json', '-show_streams', '-show_format', path])
    j = json.loads(r.stdout)
    v = next((s for s in j['streams'] if s['codec_type'] == 'video'), None)
    a = next((s for s in j['streams'] if s['codec_type'] == 'audio'), None)
    return {'duration': float(j['format'].get('duration', 0)), 'video': v, 'audio': a}

def pick_encoder():
    try:
        run(['ffmpeg', '-v', 'error', '-f', 'lavfi', '-i', 'color=s=256x256:d=0.2', '-c:v', 'h264_nvenc', '-f', 'null', '-'])
        return ['-c:v', 'h264_nvenc', '-preset', 'p6', '-cq', '19', '-b:v', '0']
    except Exception:
        return ['-c:v', 'libx264', '-preset', 'medium', '-crf', '18']
VENC = pick_encoder()
print("Encoder:", VENC[1])

all_mp4 = sorted(glob.glob(os.path.join(FOLDER, '*.mp4')), key=lambda p: (re.sub(r' \(\d+\)(?=\.mp4$)', '', p), bool(re.search(r' \(\d+\)\.mp4$', p)), p))  # originales antes que 'nombre (1).mp4'
clips, seen = [], set()
for p in all_mp4:
    name = os.path.basename(p)
    if name.startswith('spot_republic') or os.path.getsize(p) == 0:
        continue
    info = probe(p)
    if info['video'] is None or info['duration'] < 1.5:
        continue
    key = (os.path.getsize(p), round(info['duration'], 2))  # copias subidas dos veces
    if key in seen:
        print("  duplicado, omitido:", name); continue
    seen.add(key)
    clips.append({'src': p, 'name': name, **info})
print(f"{len(clips)} clips únicos, {sum(c['duration'] for c in clips)/60:.1f} min de material")

HDR_TRC = {'arib-std-b67', 'smpte2084'}
def make_proxy(c, i):
    out = f"{WORK}/proxy/clip{i:02d}.mp4"
    c['proxy'] = out
    if os.path.exists(out) and os.path.getsize(out) > 0:
        return
    v = c['video']
    short = min(1440, min(v['width'], v['height']))         # más resolución = recorte 9:16 más nítido
    short -= short % 2
    scale = f"scale='if(gt(iw,ih),-2,{short})':'if(gt(iw,ih),{short},-2)':flags=lanczos"
    chains = []
    if v.get('color_transfer') in HDR_TRC:   # DJI HLG/HDR -> SDR Rec.709
        chains.append("zscale=t=linear:npl=100,format=gbrpf32le,zscale=p=bt709,"
                      "tonemap=hable:desat=0,zscale=t=bt709:m=bt709:r=tv,format=yuv420p," + scale)
    chains.append(scale + ",format=yuv420p")
    for vf in chains:
        try:
            run(['ffmpeg', '-y', '-v', 'error', '-i', c['src'], '-vf', f"{vf},fps={FPS}",
                 *VENC, '-g', str(FPS), '-c:a', 'aac', '-b:a', '192k', '-ac', '2', '-ar', '48000', out])
            return
        except RuntimeError as e:
            print("  reintentando sin tonemap:", str(e)[-200:])
    raise RuntimeError("No se pudo crear proxy de " + c['name'])

for i, c in enumerate(clips):
    print(f"proxy {i+1}/{len(clips)}: {c['name']}  ({c['duration']:.0f}s, {c['video']['width']}x{c['video']['height']}, trc={c['video'].get('color_transfer')})")
    make_proxy(c, i)
    cap = cv2.VideoCapture(c['proxy'])
    c['W'], c['H'] = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    c['nframes'] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    c['has_audio'] = c['audio'] is not None

# ---------- B-roll: videos a proxy, imágenes tal cual ----------
BROLL = []
for k, f in enumerate(BROLL_FILES):
    b = {'src': f, 'name': os.path.basename(f), 'kind': 'image' if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')) else 'video'}
    if b['kind'] == 'video':
        info = probe(f)
        if info['video'] is None: continue
        b.update(info); bc = {'src': f, 'name': b['name'], **info}
        make_proxy(bc, 100 + k); b['proxy'] = bc['proxy']
        cap = cv2.VideoCapture(b['proxy']); b['nframes'] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    else:
        img = cv2.imread(f)
        if img is None: continue
        b['img'] = img; b['duration'] = 999
    BROLL.append(b)
print(f"{len(BROLL)} b-roll listos:", ', '.join(b['name'] for b in BROLL) or '—')

# ---------- Tomas de cine (sin diálogo): proxies a 60 fps si la toma lo tiene, para cámara lenta real ----------
CINE = []
for k, f in enumerate(CINE_FILES):
    info = probe(f)
    if info['video'] is None: continue
    cc = {'src': f, 'name': os.path.basename(f), **info}
    num, den = (info['video'].get('avg_frame_rate') or '30/1').split('/')
    src_fps = float(num) / max(1.0, float(den or 1))
    cc['fps'] = 60 if src_fps >= 50 else FPS                  # a 60 fps la cámara lenta x2 es real, sin inventar cuadros
    cc['proxy'] = f"{WORK}/proxy/cine{k:02d}.mp4"
    if not os.path.exists(cc['proxy']):
        v = info['video']; short = min(1440, min(v['width'], v['height'])); short -= short % 2
        tm = ("zscale=t=linear:npl=100,format=gbrpf32le,zscale=p=bt709,tonemap=hable:desat=0,zscale=t=bt709:m=bt709:r=tv,format=yuv420p,"
              if v.get('color_transfer') in HDR_TRC else "")
        vf = f"{tm}scale='if(gt(iw,ih),-2,{short})':'if(gt(iw,ih),{short},-2)':flags=lanczos,format=yuv420p,fps={cc['fps']}"
        try:
            run(['ffmpeg', '-y', '-v', 'error', '-i', f, '-vf', vf, *VENC, '-g', str(cc['fps']), '-an', cc['proxy']])
        except RuntimeError:
            run(['ffmpeg', '-y', '-v', 'error', '-i', f, '-vf', vf.replace(tm, ''), *VENC, '-g', str(cc['fps']), '-an', cc['proxy']])
    cap = cv2.VideoCapture(cc['proxy']); cc['W'], cc['H'] = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cc['nframes'] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    CINE.append(cc)
print(f"{len(CINE)} tomas de cine listas:", ', '.join(f"{c_['name']} ({c_['fps']} fps)" for c_ in CINE) or '—')

In [ ]:
#@title 4 · IA: transcribir lo que dices en cada clip (Whisper, palabra por palabra)
def preload_cuda12_libs():
    """faster-whisper (CTranslate2) necesita cuBLAS 12 y cuDNN 9; Colab puede traer otra versión de CUDA."""
    import ctypes, sys, site
    pat = ('libcublasLt.so.12', 'libcublas.so.12', 'libcudnn*.so.9')
    roots = site.getsitepackages() + [site.getusersitepackages()]
    find = lambda: [f for r in roots for q in pat for f in glob.glob(os.path.join(r, 'nvidia', '**', q), recursive=True)]
    if not any('libcublas.so.12' in f for f in find()):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nvidia-cublas-cu12', 'nvidia-cudnn-cu12==9.*'], check=True)
    libs = sorted(set(find()), key=lambda f: ('cublasLt' not in f, 'cublas' not in f, f))
    for _ in range(3):
        pending = []
        for f in libs:
            try: ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
            except OSError: pending.append(f)
        libs = pending
        if not libs: break

def split_phrases(words, max_len=7.0):
    """Corta en frases: puntuación, pausas > 0.45 s o > max_len segundos."""
    out, cur = [], []
    for w in words:
        if cur and (w[0] - cur[-1][1] > 0.45 or w[1] - cur[0][0] > max_len):
            out.append(cur); cur = []
        cur.append(w)
        if re.search(r'[.!?¿¡]$', w[2]) and w[1] - cur[0][0] > 1.2:
            out.append(cur); cur = []
    if cur: out.append(cur)
    return out

from faster_whisper import WhisperModel
try:
    preload_cuda12_libs()
except Exception as e:
    print("No pude preparar CUDA 12 para Whisper:", e)
def load_whisper(gpu=True):
    if gpu:
        try: return WhisperModel('large-v3', device='cuda', compute_type='float16')
        except Exception as e: print("Whisper en GPU no disponible:", str(e)[:200])
    print("Usando Whisper en CPU (más lento)")
    return WhisperModel('medium', device='cpu', compute_type='int8', cpu_threads=os.cpu_count())
wm = load_whisper(True)

def load_audio(path):
    """Lee el audio con ffmpeg (16 kHz mono) y se lo pasa a Whisper ya decodificado (evita el error de PyAV)."""
    raw = subprocess.run(['ffmpeg', '-v', 'error', '-i', path, '-f', 'f32le', '-ac', '1', '-ar', '16000', '-'],
                         capture_output=True, check=True).stdout
    return np.frombuffer(raw, np.float32).copy()

def transcribe(path):
    segs, _ = wm.transcribe(load_audio(path), language=IDIOMA, word_timestamps=True, vad_filter=True,
                            initial_prompt=f"{NOMBRE_SERVER}, servidor, server.")
    return [(w.start, w.end, w.word.strip()) for sg in segs for w in (sg.words or []) if w.word.strip()]

PHRASES = []            # todas las frases de todos los clips, en orden de grabación
for ci, c in enumerate(clips):
    c['phrases'] = []
    if not c['has_audio']:
        continue
    wav = f"{WORK}/audio/clip{ci:02d}.wav"
    run(['ffmpeg', '-y', '-v', 'error', '-i', c['proxy'], '-vn', '-ac', '1', '-ar', '16000', wav])
    try:
        words = transcribe(wav)
    except RuntimeError as e:
        print("Falló Whisper en GPU, reintento en CPU:", str(e)[:200])
        wm = load_whisper(False); words = transcribe(wav)
    for k, ph in enumerate(split_phrases(words)):
        p = {'id': f"c{ci}p{k}", 'clip': ci, 'start': ph[0][0], 'end': ph[-1][1],
             'text': ' '.join(w[2] for w in ph), 'words': ph}
        c['phrases'].append(p); PHRASES.append(p)
    print(f"  {c['name']}: {len(c['phrases'])} frases · «{' '.join(w[2] for w in words)[:110]}…»")

del wm; gc.collect()                       # libera la GPU para el modelo de IA
talk_s = sum(p['end'] - p['start'] for p in PHRASES)
print(f"\n{len(PHRASES)} frases, {talk_s:.0f} s de voz en total")
if not PHRASES:
    raise RuntimeError("No se detectó voz en los clips. Revisa IDIOMA o el audio de los videos.")
open(f"{WORK}/transcripcion.txt", 'w').write("\n".join(f"{p['id']} [{clips[p['clip']]['name']} {p['start']:.1f}s] {p['text']}" for p in PHRASES))

In [ ]:
#@title 4b · Tus órdenes al editor ("Editor, cámara lenta", "Editor, transición flash"…): se detectan, se aplican y se cortan del video
import unicodedata as _ud
def _n(s): return ''.join(ch for ch in _ud.normalize('NFD', s.lower()) if _ud.category(ch) != 'Mn')
def _clean(w): return re.sub(r'[^\w]', '', _n(w))

# Catálogo: (expresión regular sobre lo que dices después de "editor", tipo, ¿lleva texto?)
CATALOGO = [
    (r'^transicion (de )?(rayon|rayo|disco)', 'trans_rayon', False), (r'^transicion (de )?barrido', 'trans_barrido', False),
    (r'^transicion (de )?flash', 'trans_flash', False), (r'^transicion (de )?glitch', 'trans_glitch', False),
    (r'^transicion (de )?zoom', 'trans_zoom', False), (r'^transicion (a |de )?negro', 'trans_negro', False),
    (r'^(pon(le)? )?camara lenta', 'camara_lenta', False), (r'^(pon(le)? )?(zoom|acercate|acerca)\b', 'zoom', False),
    (r'^(pon(le)? )?(el )?modo comic', 'comic', False), (r'^(pon(le)? )?(el )?neon', 'neon', False),
    (r'^(pon(lo|le)? )?(en )?blanco y negro', 'bn', False), (r'^(pon(le)? )?(un )?temblor', 'temblor', False),
    (r'^desenfoca', 'desenfoca', False), (r'^(pon(lo|le)? )?(en (la )?)?(tele|tv) vieja', 'tv', False),
    (r'^silencio', 'silencio', False), (r'^bloopers?', 'bloopers', False), (r'^deja esto', 'deja', False),
    (r'^pon(le)? (un )?titulo:? (.+)', 'titulo', True), (r'^(pon(le)? )?pantalla negra:? (.+)', 'pantalla', True),
    (r'^muestra(me|les)? (los |las |el |la )?(.+)', 'muestra', True), (r'^pon(me|le|ga|gan|ganle)? la (cancion )?de (.+)', 'cancion', True),
]
ORDENES = []          # lo que se detectó, para el resumen
def parse_cmd(words):
    """words = palabras después de 'editor' → (tipo, texto, cuántas palabras usa) o None."""
    txt = ' '.join(_clean(w[2]) for w in words).strip()
    for pat, kind, takes in CATALOGO:
        m = re.match(pat, txt)
        if m:
            arg = m.groups()[-1].strip() if takes and m.groups() else ''
            used = len(m.group(0).split())
            if takes:                                   # el texto llega hasta la siguiente pausa (ya viene cortado)
                used = len(words); arg = ' '.join(w[2] for w in words[len(m.group(0).split()) - len(arg.split()):]).strip(' ,.')
            return kind, arg, used
    return None

ACTIONS = []
for ci, c in enumerate(clips):
    if not c['phrases']: continue
    words = [w for p in c['phrases'] for w in p['words']]
    keep = [True] * len(words); cmds_at = []                 # (índice de la siguiente palabra, tipo, texto, fin de la orden)
    i = 0; deja_next = False
    while i < len(words):
        if _clean(words[i][2]) == 'editor':
            j = i + 1; seg = []
            while j < len(words) and len(seg) < 8 and (not seg or words[j][0] - seg[-1][1] < 0.7):
                seg.append(words[j]); j += 1
                if re.search(r'[.!?]$', words[j - 1][2]): break        # la orden termina en el punto o la pausa
            r_ = parse_cmd(seg)
            if r_:
                kind, arg, used = r_
                end_i = i + 1 + used
                if kind == 'deja': deja_next = True
                elif not deja_next:
                    for k_ in range(i, min(end_i, len(words))): keep[k_] = False      # la orden no sale en el video
                else: deja_next = False
                if kind != 'deja':
                    cmds_at.append((end_i, kind, arg, words[min(end_i, len(words)) - 1][1]))
                    ORDENES.append((c['name'], words[i][0], kind, arg))
                i = end_i; continue
        i += 1
    if not cmds_at: continue
    # Rehace las frases del clip sin las palabras de las órdenes
    kept = [w for w, k_ in zip(words, keep) if k_]
    new_ph = []
    for k, ph in enumerate(split_phrases(kept)):
        new_ph.append({'id': f"c{ci}p{k}", 'clip': ci, 'start': ph[0][0], 'end': ph[-1][1], 'text': ' '.join(w[2] for w in ph),
                       'words': ph, 'cmds': []})
    # Cada orden se pega a lo que viene justo después; si después hay silencio (acción sin hablar), se crea un tramo de "acción"
    for end_i, kind, arg, t_end in cmds_at:
        nxt = next((p for p in new_ph if p['start'] >= t_end - 0.05), None)
        gap = (nxt['start'] if nxt else c['duration']) - t_end
        if kind in ('camara_lenta', 'zoom', 'comic', 'neon', 'bn', 'temblor', 'desenfoca', 'tv', 'cancion') and gap > 1.4:
            a_ = {'id': f"c{ci}a{len(ACTIONS)}", 'clip': ci, 'start': t_end + 0.25, 'end': min(t_end + 0.25 + 6.0, t_end + gap - 0.2),
                  'text': f"[acción sin hablar: {kind}]", 'words': [], 'cmds': [(kind, arg)], 'accion': True}
            ACTIONS.append(a_); new_ph.append(a_)
        elif nxt is not None:
            nxt['cmds'].append((kind, arg))
    new_ph.sort(key=lambda p: p['start'])
    c['phrases'] = new_ph
PHRASES = [p for c in clips for p in c['phrases']]
for p in PHRASES: p.setdefault('cmds', [])
if ORDENES:
    print(f"🎙️ {len(ORDENES)} órdenes al editor:")
    for nm, t_, kind, arg in ORDENES: print(f"   {nm} @ {t_:.1f}s → {kind}{(': ' + arg) if arg else ''}")
    print(f"   {len(ACTIONS)} tramos de acción sin hablar (para cámara lenta, etc.)")
else:
    print("No dijiste órdenes al editor (todo lo decide la IA).")

In [ ]:
#@title 5 · IA editora: arma el TRÁILER + HUMOR (tomas de cine, voz de narrador, ruptura, humor, clímax)
thumbs = []
for ci, c in enumerate(clips):
    cap = cv2.VideoCapture(c['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, c['nframes'] // 2)
    ok, fr = cap.read(); cap.release()
    if not ok: continue
    k = 640 / max(fr.shape[:2]); fr = cv2.resize(fr, None, fx=k, fy=k)
    ok, buf = cv2.imencode('.jpg', fr, [cv2.IMWRITE_JPEG_QUALITY, 80])
    thumbs.append((ci, base64.standard_b64encode(buf.tobytes()).decode()))

TONOS = {'sarcastico': "Tu voz de editor es SARCÁSTICA pero buena onda: ironía ligera, te burlas un poquito del creador o de lo obvio, nunca ofensivo ni cruel, sin groserías.",
         'gracioso': "Tu voz de editor es GRACIOSA: chistes rápidos, exageraciones, memes en palabras, sin groserías.",
         'serio': "Tu voz de editor es seria y directa, con intriga."}
cine_thumbs = []
for cc in CINE:
    cap = cv2.VideoCapture(cc['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, cc['nframes'] // 2); ok, fr = cap.read(); cap.release()
    if ok:
        k = 512 / max(fr.shape[:2]); fr = cv2.resize(fr, None, fx=k, fy=k)
        cine_thumbs.append((cc['name'], base64.standard_b64encode(cv2.imencode('.jpg', fr, [cv2.IMWRITE_JPEG_QUALITY, 80])[1].tobytes()).decode()))
cine_list = ', '.join(f"{cc['name']} ({cc['duration']:.0f}s)" for cc in CINE) or '(no hay tomas de cine)'
listing = "\n".join(f"{p['id']} | clip {p['clip']} | {p['end'] - p['start']:.1f}s | {p['text']}"
                     + (f"  [ORDEN DEL CREADOR: {', '.join(k_ + (':' + a_ if a_ else '') for k_, a_ in p['cmds'])}]" if p.get('cmds') else '')
                     for p in PHRASES)
song_names = [os.path.basename(f) for f in SONGS]
broll_names = [b['name'] for b in BROLL]
PROMPT = f"""Eres un editor de TikTok top (estilo davecast: subtítulos amarillos con la palabra clave en grande, pantallas
negras con frases del editor, mucho b-roll, cortes cada 2-3 s) con toques de Spider-Verse. Arriba hay un fotograma de cada clip.
Abajo está la transcripción de lo que dice el creador, en frases con id. Es la promo de «{NOMBRE_SERVER}»: {DESCRIPCION_SERVER}
{TONOS.get(TONO, '')}

ESTRUCTURA OBLIGATORIA "TRÁILER + HUMOR" (vertical, ~{DURACION_OBJETIVO_S} s):
1) TRÁILER (~35 % del tiempo, seccion="trailer"): épico y serio. Alterna TOMAS DE CINE sin diálogo (lista: {cine_list};
   tipo="toma", con 'archivo', 'segundos' 1.5-4, 'camara_lenta' true en las más épicas, y 'texto_trailer' = frase corta de tráiler
   en MAYÚSCULAS que aparece sobre la toma, ej. "EN UN MUNDO…", "UNA REPÚBLICA SE LEVANTÓ") con las frases donde el creador
   habla como narrador de tráiler. Puedes usar la misma toma varias veces en partes distintas.
2) RUPTURA: la primera frase de la parte de humor lleva seccion="humor"; ahí suena un rayón de disco y se rompe la seriedad.
   Lo ideal es que el tráiler termine en algo absurdo o chistoso (ej. "…y con furros") y luego la ruptura.
3) HUMOR (~50 %, seccion="humor"): su explicación con humor, zooms de risa, respuestas del editor, pantallas negras sarcásticas.
4) CIERRE (~15 %, seccion="cierre"): una toma épica de vuelta (ej. 'unirse' o 'azotea') + su frase de cierre/CTA.
Si no hay tomas de cine, el tráiler usa sus frases más épicas con b-roll.
ÓRDENES DEL CREADOR: algunas frases llevan [ORDEN DEL CREADOR: …] (cámara lenta, transición, título, etc.) y hay tramos
"[acción sin hablar …]". Esas frases y tramos SIEMPRE van en el guion, en un lugar que tenga sentido; el editor aplicará esas
órdenes automáticamente, tú no tienes que repetirlas.
Cada elemento del guion es tipo="frase" (con 'id' de la transcripción) o tipo="toma" (con 'id' = nombre del archivo de la toma).

Arma el guion de un TikTok de unos {DURACION_OBJETIVO_S} s (entre {int(DURACION_OBJETIVO_S * 0.8)} y {int(DURACION_OBJETIVO_S * 1.1)} s sumando
duraciones; si hay menos material bueno, usa lo que valga la pena) que explique con claridad qué es, qué se hace ahí (gobierno,
roles, elecciones…), por qué entrar y cómo entrar. LA GENTE TIENE QUE QUEDARSE HASTA EL FINAL: gancho fuerte en los primeros 3 s,
una intriga que se resuelve al final, capítulos claros, sin repeticiones ni tomas falsas (si una idea se dijo varias veces o en
varios lugares, elige la mejor toma). Usa SOLO frases de la lista con su id exacto; puedes reordenarlas.

Campos por frase:
- 'enfasis': la palabra o las 2 palabras más importantes de esa frase (exactas como están en la frase); van en GRANDE en el subtítulo.
- 'tarjeta': frase del editor para una pantalla negra con letras amarillas que aparece al empezar esta frase (máx. 7 palabras,
  inteligente, con el tono indicado; ej. "SÍ, OTRO SERVER… PERO ESTE TIENE PRESIDENTE", "LUEGO TE EXPLICO POR QUÉ",
  "SPOILER: NO ES BROMA"). Una cada 15-25 s; si no, "".
- 'broll': si mientras dice esa frase conviene mostrar algo, el archivo de esta lista que corresponda: {broll_names or '(no hay b-roll)'}; si no, "".
- 'estilo_broll': "pantalla" (pantalla completa), "polaroid" (foto con marco blanco) o "tv" (dentro de una TV vieja). "" si no hay broll.
- 'capitulo': título de 1-3 palabras al empezar un tema nuevo (ej. "EL GOBIERNO", "CÓMO ENTRAR"); va en letras 3D doradas. Si no, "".
- 'efecto': "spiderverse" en 2-4 momentos de máximo impacto (revelaciones, chistes, lo más épico), "neon" en 2-4 momentos para
  subrayar algo; si no, "".
- 'respuesta_editor': si en esa frase el creador le habla al editor ("avísame, editor", "editor, ponme algo"), escribe la
  RESPUESTA del editor (máx. 8 palabras, en MAYÚSCULAS, con el tono indicado; ej. "EDITOR: TE AVISÉ HACE 3 TOMAS"). Aparece en
  pantalla negra justo después de esa frase. Si no le habla al editor, "".
- 'blooper': true si la frase es un error, una toma fallida o algo que el creador llama blooper; esas frases van en una sección
  de BLOOPERS al final del video (después del cierre), así que inclúyelas en el guion. Si no, false.
- 'zoom_risa': true si al terminar esa frase hay un momento chistoso, exagerado o una reacción que merece un acercamiento de
  golpe a la cara (estilo meme). Pocas veces: 3-6 en todo el video. Si no, false.
- 'musica': cómo entra la música de fondo en esa frase: "entra" (empieza la música, normalmente justo después del gancho),
  "sube" (inicio de capítulo o momento épico: la música sube), "corte" (silencio dramático justo antes de un remate o
  revelación) o "" (sigue igual). Úsalo en momentos clave, no en todas.
- 'pedido_cancion', 'cancion', 'segundos_cancion': si en esa frase pide al editor una canción ("editor ponga la de Colombia"):
  qué pidió tal cual, el archivo de {song_names or '(no hay canciones)'} o "", y segundos (4-15). Si no: "", "", 0.
  Solo cuenta como pedido si nombra una canción concreta ("ponme la de X"); "ponme algo" NO es pedido.
  Las frases con pedido de canción SIEMPRE van en el guion.
También: 'gancho' (título 3D dorado del inicio, máx. 3 palabras), 'cta' (máx. 6 palabras; usa el link/Discord si lo dice),
'resumen' (1-2 frases).

Transcripción:
{listing}"""

STR = {'type': 'string'}
SCHEMA = {'type': 'object', 'additionalProperties': False, 'required': ['gancho', 'cta', 'resumen', 'guion'],
          'properties': {'gancho': STR, 'cta': STR, 'resumen': STR,
                         'guion': {'type': 'array', 'items': {'type': 'object', 'additionalProperties': False,
                                   'required': ['tipo', 'seccion', 'archivo', 'segundos', 'camara_lenta', 'texto_trailer', 'id', 'enfasis', 'tarjeta', 'broll', 'estilo_broll', 'capitulo', 'efecto',
                                                'respuesta_editor', 'blooper', 'zoom_risa', 'musica', 'pedido_cancion', 'cancion', 'segundos_cancion'],
                                   'properties': {'tipo': STR, 'seccion': STR, 'archivo': STR, 'segundos': {'type': 'number'},
                                                  'camara_lenta': {'type': 'boolean'}, 'texto_trailer': STR, 'id': STR, 'enfasis': STR, 'tarjeta': STR, 'broll': STR, 'estilo_broll': STR,
                                                  'capitulo': STR, 'efecto': STR, 'respuesta_editor': STR, 'blooper': {'type': 'boolean'},
                                                  'zoom_risa': {'type': 'boolean'}, 'musica': STR,
                                                  'pedido_cancion': STR, 'cancion': STR,
                                                  'segundos_cancion': {'type': 'number'}}}}}}

VLLM = {'proc': None, 'model': None}
MODELO_RESPALDO = "Qwen/Qwen3-VL-8B-Instruct"     # más chico: si el grande no cabe o no arranca
def _serve(model, util, ctx):
    import urllib.request
    vllm_bin = '/content/vllm_env/bin/vllm'
    if not os.path.exists(vllm_bin): raise RuntimeError("vLLM no está instalado (corre la celda 1 en Colab).")
    log_path = f"{WORK}/vllm.log"; log = open(log_path, 'w')
    proc = subprocess.Popen([vllm_bin, 'serve', model, '--port', '8000', '--max-model-len', str(ctx),
                             '--gpu-memory-utilization', str(util), '--limit-mm-per-prompt', json.dumps({'image': 24}),
                             '--served-model-name', 'local'], stdout=log, stderr=subprocess.STDOUT)
    print(f"Cargando {model} en la GPU (la 1.ª vez lo descarga, puede tardar 5-15 min)…"); t0 = time.time()
    while True:
        try: urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2); break
        except Exception:
            if proc.poll() is not None:
                tail = open(log_path).read()[-2500:]
                shutil.copy(log_path, os.path.join(OUT_DIR, 'vllm.log'))      # para revisarlo desde Drive
                raise RuntimeError(f"vLLM se detuvo con {model}:\n{tail}")
            if time.time() - t0 > 40 * 60:
                proc.terminate(); raise RuntimeError("vLLM tardó demasiado en arrancar")
            time.sleep(10)
    print(f"Modelo listo en {(time.time() - t0) / 60:.1f} min")
    return proc
def llm_start():
    """Arranca el modelo en la GPU una sola vez (se queda prendido para la revisión de diseño)."""
    if VLLM['proc'] is not None and VLLM['proc'].poll() is None: return
    errors = []
    for model, util, ctx in ((MODELO_LOCAL, 0.90, 32768), (MODELO_LOCAL, 0.92, 16384), (MODELO_RESPALDO, 0.6, 32768)):
        try:
            VLLM['proc'] = _serve(model, util, ctx); VLLM['model'] = model; return
        except RuntimeError as e:
            errors.append(str(e)[-600:]); print(f"⚠ No arrancó {model} (util {util}, ctx {ctx}); pruebo otra configuración…")
            subprocess.run(['pkill', '-f', 'vllm serve'], capture_output=True); time.sleep(5)
    raise RuntimeError("Ningún modelo arrancó:\n" + "\n---\n".join(errors))
def llm_stop():
    p_ = VLLM.get('proc')
    if p_ is not None and p_.poll() is None:
        p_.terminate()
        try: p_.wait(timeout=60)
        except Exception: p_.kill()
    VLLM['proc'] = None

def ask_llm(parts, schema, name='respuesta', max_tokens=16000):
    """parts = [('text', str) | ('img', base64_jpeg)] → dict validado contra el esquema JSON."""
    if MOTOR_IA == 'local_qwen':
        from openai import OpenAI
        llm_start()
        content = [{'type': 'text', 'text': v} if k == 'text' else
                   {'type': 'image_url', 'image_url': {'url': f"data:image/jpeg;base64,{v}"}} for k, v in parts]
        r = OpenAI(base_url='http://localhost:8000/v1', api_key='local').chat.completions.create(
            model='local', messages=[{'role': 'user', 'content': content}], max_tokens=max_tokens,
            temperature=0.7, top_p=0.8, presence_penalty=1.5,
            response_format={'type': 'json_schema', 'json_schema': {'name': name, 'schema': schema}},
            extra_body={'top_k': 20, 'chat_template_kwargs': {'enable_thinking': False}})
        txt = r.choices[0].message.content.strip()
        return json.loads(txt[txt.find('{'): txt.rfind('}') + 1])
    if MOTOR_IA == 'claude':
        try:
            from google.colab import userdata; key = userdata.get('ANTHROPIC_API_KEY')
        except Exception:
            key = os.environ.get('ANTHROPIC_API_KEY')
        if not key: raise RuntimeError("Falta el secreto ANTHROPIC_API_KEY")
        import anthropic
        content = [{'type': 'text', 'text': v} if k == 'text' else
                   {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': v}} for k, v in parts]
        with anthropic.Anthropic(api_key=key).beta.messages.stream(
            model='claude-opus-5-5', max_tokens=max_tokens,
            betas=['server-side-fallback-2026-07-01'], fallbacks='default',
            output_config={'effort': 'high', 'format': {'type': 'json_schema', 'schema': schema}},
            messages=[{'role': 'user', 'content': content}],
        ) as stream:
            msg = stream.get_final_message()
        if msg.stop_reason == 'refusal': raise RuntimeError("Claude declinó la petición")
        return json.loads(next(b_.text for b_ in msg.content if b_.type == 'text'))
    raise RuntimeError("MOTOR_IA = simple (sin IA)")

STOP = set("""que de la el en y a los las un una por con para es no se lo me mi te tu su al del como pero más
muy ya este esta eso esto hay son está están ser hacer tiene tienen todo todos ahí aquí entonces bueno pues o si
the and to of is it in you for on this that with are""".split())
import difflib, unicodedata
def norm(s):
    return ''.join(ch for ch in unicodedata.normalize('NFD', s.lower()) if unicodedata.category(ch) != 'Mn')
SONG_REQ = re.compile(r'\b(pon|ponga|ponle|pongan|ponme|pónganle|ponganle|metele|métele|que suene|dale play a)\b(.{0,40})', re.I)
def song_request(text):
    """Texto de la canción pedida ("la de colombia…") o '' si la frase no pide una canción concreta."""
    m = SONG_REQ.search(norm(text))
    if not m: return ''
    req = m.group(2).strip(' ,.')
    if re.search(r'\b(la de|la cancion|el tema|la rola)\b', req) or match_song(text): return req
    return ''

def match_song(text):
    """Si la frase pide una canción, devuelve el archivo de /musica que mejor coincide."""
    m = SONG_REQ.search(norm(text))
    if not m or not SONGS: return ''
    req = norm(m.group(2)); best, score = '', 0.0
    for f in SONGS:
        name = norm(re.sub(r'[_\-]+', ' ', os.path.splitext(os.path.basename(f))[0]))
        sc = max([difflib.SequenceMatcher(None, name, req).ratio()] + [1.0 for w in name.split() if len(w) > 3 and w in req])
        if sc > score: best, score = os.path.basename(f), sc
    return best if score >= 0.55 else ''

RISA = re.compile(r'\b(no mames|a la verga|pendej|jaja|jeje|miedo|que oso|no manches|tengo miedo)', re.I)
CARD_TPL = ["SÍ, OTRO SERVER… PERO CON PRESIDENTE", "SPOILER: ES EN SERIO", "AQUÍ VIENE LO BUENO",
            "POV: YA ERES CIUDADANO", "EL EDITOR NO APRUEBA ESTO", "LUEGO TE EXPLICO POR QUÉ"]
EDITOR_TPL = ["EDITOR: TE AVISÉ HACE 3 TOMAS", "EDITOR: NO ME PAGAN LO SUFICIENTE", "EDITOR: YA VOY, YA VOY",
              "EDITOR: ¿Y MI AGUINALDO?"]
CHAP_TPL = ["QUÉ ES", "CÓMO FUNCIONA", "POR QUÉ ENTRAR", "EL FINAL"]
def simple_plan():
    """Sin IA: orden de grabación con capítulos por clip, zooms en momentos de risa, pantallas sarcásticas,
    respuestas del editor, bloopers al final y música en momentos clave."""
    plan, tot, last_card, ncard, nrep, in_blooper, prev_clip, nchap = [], 0.0, -99.0, 0, 0, False, None, 0
    for k, p in enumerate(PHRASES):
        req = song_request(p['text']); txt = norm(p['text'])
        if len(p['words']) < 3 and not req and not RISA.search(txt): continue
        if 'blooper' in txt: in_blooper = True
        if prev_clip is not None and p['clip'] != prev_clip: in_blooper = 'blooper' in txt
        words = [re.sub(r'[^\wáéíóúñü]', '', w[2].lower()) for w in p['words']]
        kw = max((w for w in words if w not in STOP), key=len, default='')
        n = len(plan); new_clip = p['clip'] != prev_clip; prev_clip = p['clip']
        card = ''
        if not in_blooper and tot - last_card > 18 and n > 1: card = CARD_TPL[ncard % len(CARD_TPL)]; ncard += 1; last_card = tot
        reply = ''
        if 'editor' in txt: reply = EDITOR_TPL[nrep % len(EDITOR_TPL)]; nrep += 1
        chap = ''
        if new_clip and n > 0 and not in_blooper: chap = CHAP_TPL[min(nchap, len(CHAP_TPL) - 1)]; nchap += 1
        plan.append({'id': p['id'], 'enfasis': kw, 'tarjeta': card, 'capitulo': chap, 'respuesta_editor': reply, 'blooper': in_blooper,
                     'broll': BROLL[n // 4 % len(BROLL)]['name'] if BROLL and n % 4 == 3 and not in_blooper else '',
                     'estilo_broll': ['polaroid', 'tv', 'pantalla'][n // 4 % 3] if BROLL and n % 4 == 3 else '',
                     'efecto': 'spiderverse' if (n % 11 == 6 and not in_blooper) else ('neon' if n % 11 == 2 else ''),
                     'zoom_risa': bool(RISA.search(txt)), 'musica': 'entra' if n == 1 else ('sube' if chap else ''),
                     'pedido_cancion': req, 'cancion': match_song(p['text']), 'segundos_cancion': SEGUNDOS_CANCION_PEDIDA if req else 0})
        tot += p['end'] - p['start']
        if tot >= DURACION_OBJETIVO_S * 1.1 and not in_blooper: break
    return {'gancho': NOMBRE_SERVER.upper(), 'cta': 'ÚNETE A LA REPÚBLICA', 'resumen': 'Guion automático (sin IA).', 'guion': plan}

PLAN = None
try:
    if MOTOR_IA in ('local_qwen', 'claude'):
        parts = (sum(([('text', f"Clip {ci} (habla)"), ('img', b)] for ci, b in thumbs), []) +
                 sum(([('text', f"Toma de cine: {nm}"), ('img', b)] for nm, b in cine_thumbs), []) + [('text', PROMPT)])
        PLAN = ask_llm(parts, SCHEMA, 'guion')
except Exception as e:
    print(f"⚠ {MOTOR_IA} falló, uso el guion automático:", str(e)[-1500:])

byid = {p['id']: p for p in PHRASES}
cine_by = {cc['name']: cc for cc in CINE}
if PLAN:
    seen, ok_ = set(), []
    for g in PLAN['guion']:
        if g.get('tipo') == 'toma':
            nm = g.get('archivo') or g.get('id')
            if nm in cine_by: g['archivo'] = nm; g['id'] = nm; ok_.append(g)
        elif g['id'] in byid and g['id'] not in seen:
            seen.add(g['id']); g['tipo'] = 'frase'; ok_.append(g)
    PLAN['guion'] = ok_
if not PLAN or not PLAN['guion']:
    PLAN = simple_plan()
song_files = {os.path.basename(f): f for f in SONGS}
broll_by = {b['name']: b for b in BROLL}
DEF = (('transicion', ''), ('fx_extra', ''), ('slow_replay', False), ('zoom_inicio', False), ('tipo', 'frase'), ('seccion', 'humor'), ('archivo', ''), ('segundos', 2.5), ('camara_lenta', False), ('texto_trailer', ''),
       ('enfasis', ''), ('tarjeta', ''), ('broll', ''), ('estilo_broll', ''), ('capitulo', ''), ('efecto', ''),
       ('respuesta_editor', ''), ('blooper', False), ('zoom_risa', False), ('musica', ''),
       ('pedido_cancion', ''), ('cancion', ''), ('segundos_cancion', 0))
for g in PLAN['guion']:
    for k_, v_ in DEF: g.setdefault(k_, v_)
    if g['seccion'] not in ('trailer', 'humor', 'cierre'): g['seccion'] = 'humor'
    if g['tipo'] == 'toma':
        g['segundos'] = float(np.clip(g['segundos'] or 2.5, 1.2, 5)); g['camara_lenta'] = bool(g['camara_lenta']) and CAMARA_LENTA_IA
        for k_ in ('tarjeta', 'respuesta_editor', 'pedido_cancion', 'cancion', 'broll', 'efecto'): g[k_] = ''
        g['zoom_risa'] = False; g['blooper'] = False; g['segundos_cancion'] = 0.0
        continue
    txt_ = byid[g['id']]['text']
    if not g['pedido_cancion'].strip(): g['pedido_cancion'] = song_request(txt_)
    if g['cancion'] not in song_files: g['cancion'] = match_song(txt_)
    if MODO_CANCIONES != 'usar_archivos': g['cancion'] = ''
    g['segundos_cancion'] = float(SEGUNDOS_CANCION_PEDIDA) if g['pedido_cancion'] else 0.0
    if g['broll'] not in broll_by: g['broll'] = ''
    if g['broll'] and g['estilo_broll'] not in ('pantalla', 'polaroid', 'tv'): g['estilo_broll'] = 'pantalla'
    if g['estilo_broll'] == 'tv' and not TV_VIEJA: g['estilo_broll'] = 'polaroid'
    if g['efecto'] == 'spiderverse' and not SPIDERVERSE: g['efecto'] = ''
    if g['efecto'] == 'neon' and not NEON: g['efecto'] = ''
    if not ZOOMS_RISA: g['zoom_risa'] = False
    if g['musica'] not in ('entra', 'sube', 'corte') or not MUSICA_MOMENTOS_CLAVE: g['musica'] = ''
have = {g['id'] for g in PLAN['guion'] if g['tipo'] == 'frase'}
for p in PHRASES:
    req = song_request(p['text'])
    if req and p['id'] not in have:
        print(f"  + añado el pedido de canción «{p['text']}»")
        pos = next((i for i, g in enumerate(PLAN['guion']) if g['tipo'] == 'frase' and PHRASES.index(byid[g['id']]) > PHRASES.index(p)), len(PLAN['guion']))
        g = dict(DEF); g.update({'id': p['id'], 'pedido_cancion': req, 'segundos_cancion': float(SEGUNDOS_CANCION_PEDIDA),
                                 'cancion': match_song(p['text']) if MODO_CANCIONES == 'usar_archivos' else ''})
        PLAN['guion'].insert(pos, g)
# ---------- Tus órdenes mandan: se incluyen siempre y pisan lo que haya decidido la IA ----------
have = {g['id'] for g in PLAN['guion'] if g['tipo'] == 'frase'}
for p in PHRASES:
    if (p.get('cmds') or p.get('accion')) and p['id'] not in have:
        pos = next((i for i, g in enumerate(PLAN['guion']) if g['tipo'] == 'frase' and PHRASES.index(byid[g['id']]) > PHRASES.index(p)), len(PLAN['guion']))
        g = dict(DEF); g.update({'id': p['id'], 'seccion': PLAN['guion'][pos - 1]['seccion'] if pos else 'humor'}); PLAN['guion'].insert(pos, g)
def _best(name, options):
    name = norm(name); best, sc = None, 0.0
    for o in options:
        o_ = norm(re.sub(r'[_\-]+', ' ', os.path.splitext(o)[0]))
        s_ = max([difflib.SequenceMatcher(None, name, o_).ratio()] + [1.0 for w in o_.split() if len(w) > 3 and w in name])
        if s_ > sc: best, sc = o, s_
    return best if sc >= 0.5 else None
blooper_from = {}
for g in PLAN['guion']:
    if g['tipo'] != 'frase': continue
    p = byid[g['id']]
    for kind, arg in p.get('cmds', []):
        if kind.startswith('trans_'): g['transicion'] = kind[6:]
        elif kind == 'camara_lenta': g['slow_replay'] = True
        elif kind == 'zoom': g['zoom_inicio'] = True
        elif kind == 'comic': g['efecto'] = 'spiderverse'
        elif kind == 'neon': g['efecto'] = 'neon'
        elif kind in ('bn', 'temblor', 'desenfoca', 'tv'): g['fx_extra'] = kind
        elif kind == 'silencio': g['musica'] = 'corte'
        elif kind == 'titulo' and arg: g['capitulo'] = arg.upper()
        elif kind == 'pantalla' and arg: g['tarjeta'] = arg.upper()
        elif kind == 'muestra':
            b_ = _best(arg, list(broll_by))
            if b_: g['broll'] = b_; g['estilo_broll'] = 'tv' if re.search(r'\b(tele|tv)\b', norm(arg)) else (g['estilo_broll'] or 'pantalla')
            else: print(f"  ⚠ 'muestra {arg}': no encontré ese b-roll en {BROLL_DIR}")
        elif kind == 'cancion':
            g['pedido_cancion'] = arg; g['segundos_cancion'] = float(SEGUNDOS_CANCION_PEDIDA)
            g['cancion'] = (_best(arg, list(song_files)) or '') if MODO_CANCIONES == 'usar_archivos' else ''
        elif kind == 'bloopers': blooper_from[p['clip']] = min(blooper_from.get(p['clip'], 1e9), p['start'])
    if p.get('accion'): g['enfasis'] = ''; g['tarjeta'] = g['tarjeta'] if g['tarjeta'] else ''
for g in PLAN['guion']:
    if g['tipo'] == 'frase' and byid[g['id']]['clip'] in blooper_from and byid[g['id']]['start'] >= blooper_from[byid[g['id']]['clip']]:
        g['blooper'] = True
# "Editor, transición rayón": ahí termina el tráiler y empieza el humor
k_r = next((i for i, g in enumerate(PLAN['guion']) if g.get('transicion') == 'rayon'), None)
if k_r is not None:
    for i, g in enumerate(PLAN['guion']):
        if i < k_r: g['seccion'] = 'trailer'
        elif g['seccion'] == 'trailer': g['seccion'] = 'humor'
if CINE and not any(g['tipo'] == 'toma' for g in PLAN['guion']) and any(g['seccion'] == 'trailer' for g in PLAN['guion']):
    # sin IA: intercala tus tomas de cine entre las frases del tráiler
    tpl = iter(["EN UN MUNDO…", "UNA REPÚBLICA SE LEVANTÓ", "NADA SERÁ IGUAL", NOMBRE_SERVER.upper()])
    out_, k_c = [], 0
    for g in PLAN['guion']:
        if g['seccion'] == 'trailer' and g['tipo'] == 'frase' and k_c < len(CINE) * 2:
            cc = CINE[k_c % len(CINE)]; k_c += 1
            out_.append({**dict(DEF), 'tipo': 'toma', 'id': cc['name'], 'archivo': cc['name'], 'seccion': 'trailer', 'segundos': 2.2,
                         'camara_lenta': CAMARA_LENTA_IA and k_c == 1, 'texto_trailer': next(tpl, '')})
            for b_k in ('broll', 'estilo_broll', 'tarjeta'): g[b_k] = ''
        out_.append(g)
    PLAN['guion'] = out_
PLAN['guion'] = [g for g in PLAN['guion'] if not g['blooper']] + [g for g in PLAN['guion'] if g['blooper']]   # bloopers al final
if PLAN['guion'] and all(g['seccion'] != 'trailer' for g in PLAN['guion']) and CINE:      # sin tráiler: arma uno corto
    intro = [{**dict(DEF), 'tipo': 'toma', 'id': cc['name'], 'archivo': cc['name'], 'seccion': 'trailer', 'segundos': 2.4,
              'camara_lenta': CAMARA_LENTA_IA and k == 0, 'texto_trailer': t_} for k, (cc, t_) in
             enumerate(zip(CINE[:3], ["EN UN MUNDO…", "UNA REPÚBLICA SE LEVANTÓ", NOMBRE_SERVER.upper()]))]
    PLAN['guion'] = intro + PLAN['guion']
def _glen(g): return g['segundos'] * (1.0 if not g['camara_lenta'] else 1.0) if g['tipo'] == 'toma' else byid[g['id']]['end'] - byid[g['id']]['start'] + g['segundos_cancion']
def _len(): return sum(_glen(g) for g in PLAN['guion'])
while len(PLAN['guion']) > 3 and _len() > DURACION_OBJETIVO_S * 1.25:
    drop = next((i for i in range(len(PLAN['guion']) - 2, 0, -1) if PLAN['guion'][i]['tipo'] == 'frase' and PLAN['guion'][i]['seccion'] == 'humor'
                 and not PLAN['guion'][i]['pedido_cancion'] and not PLAN['guion'][i]['blooper']), None)
    if drop is None: break
    PLAN['guion'].pop(drop)
if TEXTO_CTA.strip(): PLAN['cta'] = TEXTO_CTA.strip()

print(f"Gancho: {PLAN['gancho']}  ·  CTA: {PLAN['cta']}  ·  {len(PLAN['guion'])} frases, ~{_len():.0f}s\n")
last_sec = None
for g in PLAN['guion']:
    if g['seccion'] != last_sec:
        print({'trailer': "\n  ═══ 🎬 TRÁILER ═══", 'humor': "\n  ═══ 💿 *rayón* → HUMOR ═══", 'cierre': "\n  ═══ 🏁 CIERRE ═══"}[g['seccion']]); last_sec = g['seccion']
    if g['tipo'] == 'toma':
        print(f"  🎞 TOMA {g['archivo']} {g['segundos']:.1f}s{' 🐢 cámara lenta' if g['camara_lenta'] else ''}  «{g['texto_trailer']}»"); continue
    if g['capitulo']: print(f"\n  ── ✨ {g['capitulo'].upper()} ──")
    if g['tarjeta']: print(f"  ⬛ {g['tarjeta'].upper()}")
    if g['blooper'] and not any(x['blooper'] for x in PLAN['guion'][:PLAN['guion'].index(g)]): print("\n  ── 🎬 BLOOPERS ──")
    extra = ' '.join(x for x in (f"🎞 {g['broll']} ({g['estilo_broll']})" if g['broll'] else '',
                                 {'spiderverse': '🕷 SPIDER-VERSE', 'neon': '💡 NEÓN'}.get(g['efecto'], ''),
                                 '😂 ZOOM' if g['zoom_risa'] else '', {'entra': '🎶 entra', 'sube': '🎶⬆ sube', 'corte': '🔇 corte'}.get(g['musica'], ''),
                                 f"🎵 {g['pedido_cancion']} → {g['cancion'] or 'TikTok'}" if g['pedido_cancion'] else '') if x)
    print(f"  {byid[g['id']]['text']}   [{g['enfasis']}]  {extra}")
    if g['respuesta_editor']: print(f"  🗨 {g['respuesta_editor'].upper()}")
    mine = [k_ for k_, _ in byid[g['id']].get('cmds', [])]
    if mine: print(f"  🎙️ tu orden: {', '.join(mine)}")

In [ ]:
#@title 6 · Línea de tiempo CINE: tráiler (tomas + narrador) → rayón → humor → cierre, música de tráiler y diseño de sonido
HOOK_S, CTA_S, CARD_S = 2.6, 3.0, 1.7
globals().pop('BLOOPER_F0', None)
PIECES = []
for g in PLAN['guion']:
    if g['tipo'] == 'toma':
        PIECES.append({'kind': 'cine', 'cc': cine_by[g['archivo']], 'dur': g['segundos'], 'slow': bool(g['camara_lenta']),
                       'texto': g['texto_trailer'].strip().upper(), 'seccion': g['seccion'], 'phr': [], 'req': '', 'song': None,
                       'hold': 0.0, 'blooper': False, 'clip': None})
        continue
    p = byid[g['id']]; c = clips[p['clip']]
    if p.get('accion') and g['slow_replay'] is False and any(k_ == 'camara_lenta' for k_, _ in p.get('cmds', [])):
        g['slow_replay'] = True
    if p.get('accion') and g['slow_replay']:                 # acción sin hablar en cámara lenta (p. ej. caminando)
        span = p['end'] - p['start']
        PIECES.append({'kind': 'cine', 'cc': {'proxy': c['proxy'], 'fps': FPS, 'nframes': c['nframes'], 'name': c['name']},
                       'dur': min(6.0, span * 2.5), 'slow': True, 'texto': '', 'seccion': g['seccion'], 'phr': [], 'req': '',
                       'song': None, 'hold': 0.0, 'blooper': bool(g['blooper']), 'clip': None, 'src_fixed': int(p['start'] * FPS),
                       'trans': g['transicion'], 'fx_extra': g['fx_extra']})
        continue
    s0 = max(0.0, p['start'] - 0.08); s1 = min(c['duration'], p['end'] + 0.15)
    song = song_files.get(g['cancion']) if g['cancion'] else None
    req = g['pedido_cancion'].strip(); hold = float(g['segundos_cancion']) if req else 0.0
    prev = PIECES[-1] if PIECES else None
    if (prev and prev['kind'] == 'talk' and not prev['req'] and not req and prev['clip'] == p['clip'] and 0 <= s0 - prev['s1'] < 0.3
            and prev['blooper'] == bool(g['blooper']) and prev['seccion'] == g['seccion'] and not g['transicion']
            and not prev.get('replay_after')):
        prev['s1'] = s1; prev['phr'].append((p, g))
    else:
        PIECES.append({'kind': 'talk', 'clip': p['clip'], 's0': s0, 's1': s1, 'phr': [(p, g)], 'song': song, 'req': req, 'hold': hold,
                       'blooper': bool(g['blooper']), 'seccion': g['seccion'], 'trans': g['transicion']})
    if g['slow_replay'] and not p.get('accion'):            # repetición en cámara lenta del final de la frase
        PIECES[-1]['replay_after'] = True
        PIECES.append({'kind': 'cine', 'cc': {'proxy': c['proxy'], 'fps': FPS, 'nframes': c['nframes'], 'name': c['name']},
                       'dur': 3.2, 'slow': True, 'texto': '', 'seccion': g['seccion'], 'phr': [], 'req': '', 'song': None,
                       'hold': 0.0, 'blooper': bool(g['blooper']), 'clip': None, 'src_fixed': int(max(0, p['end'] - 1.3) * FPS),
                       'trans': '', 'fx_extra': ''})

t = 0
WORDS, CAPS, CARDS, BROLLS, TITLES, FX, SONG_EVENTS, SONG_MARKS, SFX = [], [], [], [], [], [], [], [], []
ZOOMS, MCUES, TRAILER_TXT, TRANS = [], [], [], []
for k, pc in enumerate(PIECES):
    pc['idx'] = k
    if pc['kind'] == 'cine':                               # toma de cine: sin voz, con texto de tráiler
        cc = pc['cc']; pc['n'] = pc['n_talk'] = max(FPS, int(round(pc['dur'] * FPS)))
        need = pc['n'] * (cc['fps'] / FPS) / (2.5 if pc['slow'] else 1)        # cuadros de origen que se consumen
        pc['src_f0'] = pc['src_fixed'] if 'src_fixed' in pc else int(max(0, min(cc['nframes'] - need - 2, cc['nframes'] * 0.2)))
        pc['f0'] = t; pc['f1'] = t + pc['n']; t = pc['f1']
        if pc['texto']: TRAILER_TXT.append({'f0': pc['f0'] + int(0.25 * FPS), 'f1': pc['f1'] - 3, 'text': pc['texto']})
        if pc.get('trans'): TRANS.append({'kind': pc['trans'], 'f': pc['f0']})
        if pc.get('fx_extra'): FX.append({'kind': pc['fx_extra'], 'f0': pc['f0'], 'f1': pc['f1']})
        if 'src_fixed' in pc: SFX.append(('whoosh', pc['f0'] / FPS))
        continue
    pc['src_f0'] = int(round(pc['s0'] * FPS))
    pc['n_talk'] = max(1, int(round((pc['s1'] - pc['s0']) * FPS)))
    pc['n'] = pc['n_talk'] + int(round(pc['hold'] * FPS))
    pc['f0'] = t; pc['f1'] = t + pc['n']; t = pc['f1']
    if pc.get('trans'): TRANS.append({'kind': pc['trans'], 'f': pc['f0']})
    if pc['req']:
        ev = {'file': pc['song'], 'name': pc['req'], 't0': (pc['f0'] + pc['n_talk']) / FPS, 'dur': pc['hold']}
        (SONG_EVENTS if pc['song'] else SONG_MARKS).append(ev)
    for p, g in pc['phr']:
        off = pc['f0'] / FPS - pc['s0']
        ws = [(a + off, b + off, w) for a, b, w in p['words']]; WORDS += ws
        emph = {norm(re.sub(r'[^\w]', '', x)) for x in g['enfasis'].split()}
        # subtítulos: grupos de 2-5 palabras; la de énfasis va grande
        grp = []
        for w in ws:
            if grp and (len(grp) >= 5 or w[0] - grp[-1][1] > 0.45 or (len(grp) >= 2 and grp[-1][2].endswith((',', '.', '?', '!')))):
                CAPS.append(grp); grp = []
            grp.append((w[0], w[1], w[2], norm(re.sub(r'[^\w]', '', w[2])) in emph))
        if grp: CAPS.append(grp)
        fs, fe = int((p['start'] + off) * FPS), int((p['end'] + off) * FPS)
        if g['tarjeta'].strip():
            CARDS.append({'f0': fs, 'f1': fs + int(CARD_S * FPS), 'text': g['tarjeta'].strip().upper()})
        if g['respuesta_editor'].strip():                  # el editor le contesta (justo al terminar la frase)
            CARDS.append({'f0': fe + 2, 'f1': fe + 2 + int(1.5 * FPS), 'text': g['respuesta_editor'].strip().upper(), 'reply': True})
        if g['blooper'] and 'BLOOPER_F0' not in globals():
            BLOOPER_F0 = pc['f0']                          # los bloopers siempre empiezan plano nuevo
        if g['broll']:
            b0 = fs + (int(CARD_S * FPS) if g['tarjeta'].strip() else int(0.25 * FPS))
            b1 = max(b0 + int(2.2 * FPS), min(fe, b0 + int(5 * FPS)))
            BROLLS.append({'f0': b0, 'f1': min(b1, pc['f1']), 'b': broll_by[g['broll']], 'style': g['estilo_broll']})
        if g['capitulo'].strip() and TITULOS_3D_DORADOS:
            TITLES.append({'f0': fs, 'f1': fs + int(2.4 * FPS), 'text': g['capitulo'].strip().upper(), 'piece': k, 'big': True})
        if g['zoom_risa']:
            z0 = max(pc['f0'], fe - int(0.3 * FPS))
            ZOOMS.append({'f0': z0, 'f1': z0 + int(1.2 * FPS), 'piece': k})      # puede seguir en el plano siguiente
        if g['musica']:
            MCUES.append({'kind': g['musica'], 't0': fs / FPS, 't1': fe / FPS})
        if g['zoom_inicio']: ZOOMS.append({'f0': fs, 'f1': fs + int(1.4 * FPS), 'piece': k})
        if g['fx_extra']: FX.append({'kind': g['fx_extra'], 'f0': fs, 'f1': fs + int(2.6 * FPS)})
        if g['efecto'] in ('spiderverse', 'neon'):
            dur = min(3.0, max(1.4, (fe - fs) / FPS)) if g['efecto'] == 'spiderverse' else 1.6
            FX.append({'kind': g['efecto'], 'f0': fs, 'f1': fs + int(dur * FPS)})
TOTAL_FRAMES = t; L = TOTAL_FRAMES / FPS
RUPT_F = next((p_['f0'] for p_ in PIECES if p_['seccion'] != 'trailer'), None) if PIECES[0]['seccion'] == 'trailer' else None
if RUPT_F is not None and RUPT_F <= 0: RUPT_F = None
CIERRE_F = next((p_['f0'] for p_ in PIECES if p_['seccion'] == 'cierre'), None)
if TITULOS_3D_DORADOS and PIECES[0]['kind'] == 'talk':
    TITLES.insert(0, {'f0': 0, 'f1': int(min(HOOK_S, PIECES[0]['n'] / FPS) * FPS), 'text': PLAN['gancho'].upper(), 'piece': 0, 'big': True})
MAIN_END = globals().get('BLOOPER_F0', TOTAL_FRAMES)           # el cierre va antes de los bloopers
CTA = {'f0': max(0, MAIN_END - int(CTA_S * FPS)), 'f1': MAIN_END, 'text': PLAN['cta'].upper()}
if MAIN_END < TOTAL_FRAMES:
    CARDS = [c_ for c_ in CARDS if not (MAIN_END - FPS * 2 <= c_['f0'] < MAIN_END + FPS * 2)]
    CARDS.append({'f0': MAIN_END, 'f1': MAIN_END + int(1.4 * FPS), 'text': 'BLOOPERS'})
for w in TITLES:
    pc = PIECES[w['piece']]; w['f0'] = max(w['f0'], pc['f0']); w['f1'] = min(w['f1'], pc['f1'])
if RUPT_F:                                                    # en el tráiler no hay tarjetas sarcásticas ni efectos de humor
    CARDS = [c_ for c_ in CARDS if c_['f0'] >= RUPT_F]; ZOOMS = [z for z in ZOOMS if z['f0'] >= RUPT_F]
    FX = [x for x in FX if x['f0'] >= RUPT_F or x['kind'] in ('neon', 'bn', 'temblor', 'desenfoca', 'tv')]
TITLES = [w for w in TITLES if w['f1'] - w['f0'] >= 15]
# nada encima de nada: tarjetas mandan; luego b-roll; títulos 3D solo sobre la persona hablando
def clash(a, lst, pad=0): return any(a['f0'] < b['f1'] + pad and b['f0'] < a['f1'] + pad for b in lst)
CARDS.sort(key=lambda c_: c_['f0'])
CARDS = [c_ for i, c_ in enumerate(CARDS) if not clash(c_, CARDS[:i], FPS * (1 if c_.get('reply') else 4))
         and (c_['f1'] < CTA['f0'] or c_['f0'] >= MAIN_END)]
BROLLS = [b for i, b in enumerate(BROLLS) if not clash(b, CARDS) and not clash(b, BROLLS[:i]) and (b['f1'] <= CTA['f0'] or b['f0'] >= MAIN_END)]
TITLES = [w for w in TITLES if not clash(w, CARDS) and not clash(w, BROLLS)]
FX = [x for i, x in enumerate(FX) if x['kind'] in ('bn', 'temblor', 'desenfoca', 'tv') or (not clash(x, CARDS) and not clash(x, FX[:i], FPS * 3))]
for z in ZOOMS: z['f1'] = min(z['f1'], TOTAL_FRAMES)
ZOOMS = [z for i, z in enumerate(ZOOMS) if not clash(z, CARDS) and not clash(z, BROLLS) and not clash(z, ZOOMS[:i], FPS * 4)]

# ritmo: cambio de encuadre cada ~2.5 s (alineado a palabra) como en davecast
for pc in PIECES:
    starts = sorted(int(round(a * FPS)) - pc['f0'] for a, _, _ in WORDS if pc['f0'] <= a * FPS < pc['f0'] + pc['n_talk'])
    marks, last = [0], 0
    for s_ in starts:
        if s_ - last >= int(2.4 * FPS) and pc['n'] - s_ >= int(1.2 * FPS): marks.append(s_); last = s_
    pc['zmarks'] = marks
    lv = [1.0, 1.16, 1.06, 1.24] if pc['idx'] % 2 == 0 else [1.12, 1.0, 1.2, 1.05]
    pc['zlevels'] = [lv[j % 4] for j in range(len(marks))]
    pc['xoff'] = [(-0.04, 0.0, 0.04, 0.0)[j % 4] for j in range(len(marks))]   # leve cambio de lado = "otro ángulo"

GROSERIAS = re.compile(r'^(puta|puto|putas|putos|verga|vergas|pendejo|pendeja|pendejos|cabron|cabrona|cabrones|chingar|chinga|'
                       r'chingada|chingado|chingadera|mierda|culero|culera|joder|marica|hijueputa|pinche|pinches|mamada|mamadas)$')
BEEPS = []
if CENSURAR_GROSERIAS:
    def censor(w): return '#$%&' if GROSERIAS.match(norm(re.sub(r'[^\w]', '', w))) else w
    for grp in CAPS:
        for j_, w_ in enumerate(grp):
            if censor(w_[2]) != w_[2]: grp[j_] = (w_[0], w_[1], '#$%&', w_[3]); BEEPS.append((w_[0], w_[1]))
    print(f"  censura: {len(BEEPS)} groserías tapadas")
for x in CARDS: SFX.append(('whoosh', x['f0'] / FPS))
for x in BROLLS: SFX.append(('whoosh', x['f0'] / FPS))
for x in TITLES: SFX.append(('hit', x['f0'] / FPS))
for x in FX: SFX.append(('glitch' if x['kind'] == 'spiderverse' else 'shine', x['f0'] / FPS))
for e in SONG_EVENTS + SONG_MARKS: SFX.append(('riser', e['t0'] - 0.9))
for z in ZOOMS: SFX.append(('boom', z['f0'] / FPS))
TSFX = {'rayon': 'scratch', 'barrido': 'whoosh', 'flash': 'hit', 'glitch': 'glitch', 'zoom': 'whoosh', 'negro': None}
for tr in TRANS:
    if TSFX.get(tr['kind']): SFX.append((TSFX[tr['kind']], tr['f'] / FPS - 0.08))
    if tr['kind'] == 'rayon': MCUES.append({'kind': 'corte', 't0': tr['f'] / FPS - 0.1, 't1': tr['f'] / FPS + 0.5})
for x in FX:
    if x['kind'] == 'temblor': SFX.append(('boom', x['f0'] / FPS))
if DISENO_SONIDO_CINE:
    for j_, x in enumerate(TRAILER_TXT): SFX.append(('braam' if j_ % 2 == 0 else 'hit', x['f0'] / FPS))
    if RUPT_F: SFX.append(('scratch', RUPT_F / FPS - 0.05))
    if CIERRE_F: SFX.append(('braam', CIERRE_F / FPS))
for m in MCUES:
    if m['kind'] == 'entra': SFX.append(('hit', m['t0']))
    if m['kind'] == 'corte': SFX.append(('hit', m['t1']))
print(f"{len(PIECES)} planos ({sum(p_['kind'] == 'cine' for p_ in PIECES)} tomas de cine) · tráiler hasta {((RUPT_F or 0) / FPS):.1f}s · {L / 60:.1f} min · {len(CAPS)} subtítulos · {len(CARDS)} pantallas negras · {len(BROLLS)} b-roll · "
      f"{len(TITLES)} títulos 3D · {sum(x['kind'] == 'spiderverse' for x in FX)} Spider-Verse · {sum(x['kind'] == 'neon' for x in FX)} neón · "
      f"{len(ZOOMS)} zooms · {len(TRANS)} transiciones · {len(MCUES)} momentos de música · {sum(1 for c_ in CARDS if c_.get('reply'))} respuestas del editor · "
      f"{'bloopers ✓ · ' if MAIN_END < TOTAL_FRAMES else ''}"
      f"{len(SONG_EVENTS)} canciones dentro · {len(SONG_MARKS)} para TikTok")

# ---------- Audio ----------
SR = 48000
parts = []
for pc in PIECES:
    d = pc['n'] / FPS; dt = pc['n_talk'] / FPS; p = f"{WORK}/audio/piece{pc['idx']:03d}.wav"
    c = clips[pc['clip']] if pc['kind'] == 'talk' else {'has_audio': False}          # tomas de cine: sin voz
    src = ['-f', 'lavfi', '-i', 'anullsrc=r=48000:cl=stereo'] if not c['has_audio'] else ['-ss', f"{pc['src_f0'] / FPS:.3f}", '-i', c['proxy']]
    duck = f",volume='1-0.75*clip((t-{dt:.3f})/0.4,0,1)':eval=frame" if pc['req'] else ""
    run(['ffmpeg', '-y', '-v', 'error', *src, '-vn', '-ac', '2', '-ar', str(SR),
         '-af', f"apad,atrim=0:{d:.4f}{duck},afade=t=in:d=0.02,afade=t=out:st={max(0, d - 0.03):.4f}:d=0.03", p])
    parts.append(p)
open(f"{WORK}/alist.txt", 'w').writelines(f"file '{p}'\n" for p in parts)
VOICE_WAV = f"{WORK}/voz.wav"
run(['ffmpeg', '-y', '-v', 'error', '-f', 'concat', '-safe', '0', '-i', f"{WORK}/alist.txt", '-c', 'copy', VOICE_WAV])
VOICE_PROC = f"{WORK}/voz_limpia.wav"
run(['ffmpeg', '-y', '-v', 'error', '-i', VOICE_WAV, '-af',
     "highpass=f=80,afftdn=nf=-25,acompressor=threshold=-20dB:ratio=3:attack=5:release=80,"
     f"loudnorm=I=-14:TP=-1.5:LRA=7,aresample={SR},apad,atrim=0:{L:.4f}", '-ar', str(SR), VOICE_PROC])

# Efectos de sonido sintetizados (whoosh, golpe, glitch, brillo, subida antes de la canción)
def synth(kind):
    rng = np.random.default_rng({'whoosh': 1, 'hit': 2, 'glitch': 3, 'shine': 4, 'riser': 5, 'boom': 6, 'braam': 7, 'scratch': 8}[kind])
    if kind == 'riser':
        n = int(0.9 * SR); tt = np.arange(n) / SR; nz = rng.standard_normal(n)
        tone = np.sin(2 * np.pi * (200 * tt + 900 * tt ** 2)) * 0.25
        return ((nz * 0.15 + tone) * (tt / 0.9) ** 2).astype(np.float32)
    if kind == 'braam':                                    # "BRAAAM" de tráiler: metales graves saturados
        n = int(2.6 * SR); tt = np.arange(n) / SR
        x = sum(np.sign(np.sin(2 * np.pi * f * (1 + 0.002 * np.sin(tt * 3)) * tt)) * a for f, a in ((41, 1), (61.5, 0.7), (82, 0.5), (123, 0.25)))
        y = np.zeros(n, np.float32); acc = 0.0; a_ = 0.06
        for i in range(n): acc += a_ * (x[i] - acc); y[i] = acc            # filtro paso bajo
        env = np.minimum(1, tt / 0.04) * np.exp(-tt * 1.1)
        return (np.tanh(y * 2.2) * env * 0.75).astype(np.float32)
    if kind == 'scratch':                                  # rayón de disco
        n = int(0.55 * SR); tt = np.arange(n) / SR
        f_ = 900 + 700 * np.sin(2 * np.pi * 5.5 * tt) * np.exp(-tt * 2)
        x = np.sin(2 * np.pi * np.cumsum(f_) / SR) * 0.4 + rng.standard_normal(n) * 0.25
        return (x * np.exp(-tt * 4) * (np.abs(np.sin(2 * np.pi * 9 * tt)) > 0.2)).astype(np.float32)
    if kind == 'boom':                                     # "boom" grave de meme
        n = int(1.0 * SR); tt = np.arange(n) / SR
        f_ = 70 * np.exp(-tt * 1.6) + 35
        x = np.sin(2 * np.pi * np.cumsum(f_) / SR) * np.exp(-tt * 3.2)
        return (np.tanh(x * 3) * 0.8).astype(np.float32)
    if kind == 'hit':
        n = int(0.6 * SR); tt = np.arange(n) / SR
        return (np.sin(2 * np.pi * 55 * tt) * np.exp(-tt * 7) * 0.9 + rng.standard_normal(n) * np.exp(-tt * 30) * 0.3).astype(np.float32)
    if kind == 'glitch':
        n = int(0.35 * SR); x = rng.standard_normal(n) * 0.3
        x *= np.repeat(rng.random(n // 600 + 1) > 0.4, 600)[:n]
        return (np.sign(x) * np.abs(x) ** 0.5 * 0.35).astype(np.float32)
    if kind == 'shine':
        n = int(0.7 * SR); tt = np.arange(n) / SR
        return (sum(np.sin(2 * np.pi * f * tt) for f in (1760, 2637, 3520)) / 3 * np.exp(-tt * 5) * 0.25).astype(np.float32)
    n = int(0.55 * SR); tt = np.arange(n) / SR; nz = rng.standard_normal(n)
    env = np.sin(np.pi * tt / tt[-1]) ** 2
    out = np.zeros(n, np.float32); y = 0.0
    for i in range(n):                                   # filtro paso bajo que barre (el "fiuuu")
        a_ = 0.02 + 0.5 * env[i]; y += a_ * (nz[i] - y); out[i] = y
    return out * env * 0.6
SFX_WAV = f"{WORK}/sfx.wav"
sfx_track = np.zeros((int(L * SR) + SR, ), np.float32)
if EFECTOS_SONIDO:
    bank = {}
    for kind, t0 in SFX:
        if t0 < 0: continue
        snd = bank.setdefault(kind, synth(kind)); i0 = int(t0 * SR)
        sfx_track[i0:i0 + len(snd)] += snd[:max(0, len(sfx_track) - i0)]
import wave
with wave.open(SFX_WAV, 'wb') as wf:
    wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(SR)
    wf.writeframes((np.clip(sfx_track[:int(L * SR)], -1, 1) * 32767 * 0.5).astype(np.int16).tobytes())

# Música de fondo: lista en orden con fundidos largos
def find_song(name):
    want = norm(re.sub(r'[_\-]+', ' ', os.path.splitext(name.strip())[0])); best, score = None, 0.0
    for f in SONGS:
        have_ = norm(re.sub(r'[_\-]+', ' ', os.path.splitext(os.path.basename(f))[0]))
        sc = 1.0 if want == have_ else difflib.SequenceMatcher(None, want, have_).ratio()
        if sc > score: best, score = f, sc
    return best if score >= 0.6 else None
BG_FILES = []
for name in [n for n in MUSICA_FONDO.split(',') if n.strip()]:
    f = find_song(name)
    print(f"  fondo: «{name.strip()}» → {os.path.basename(f) if f else 'NO ENCONTRADA'}")
    if f: BG_FILES.append(f)
BG_WAV = f"{WORK}/fondo.wav"
if BG_FILES:
    ins = sum((['-i', f] for f in BG_FILES), [])
    chain = [';'.join(f"[{i}:a]aresample={SR},aformat=channel_layouts=stereo[b{i}]" for i in range(len(BG_FILES)))]; last = '[b0]'
    for i in range(1, len(BG_FILES)):
        chain.append(f"{last}[b{i}]acrossfade=d=4:c1=qsin:c2=qsin[x{i}]"); last = f'[x{i}]'
    run(['ffmpeg', '-y', '-v', 'error', *ins, '-filter_complex', ';'.join(chain), '-map', last, '-ar', str(SR), BG_WAV])
else:
    run(['ffmpeg', '-y', '-v', 'error', '-f', 'lavfi', '-i', f'anullsrc=r={SR}:cl=stereo', '-t', '1', BG_WAV])

def music_gain(t):
    """Volumen de la música de fondo en cada instante: entra tras el gancho, sube en momentos épicos,
    se corta antes de un remate y baja suave alrededor de las canciones pedidas."""
    g_ = np.ones_like(t)
    if MUSICA_MOMENTOS_CLAVE:
        ent = [m['t0'] for m in MCUES if m['kind'] == 'entra']
        start = (RUPT_F / FPS + 0.1) if RUPT_F else (ent[0] if ent else min(HOOK_S, L * 0.1))
        g_ *= np.clip((t - start) / 0.25, 0, 1)
        for m in MCUES:
            if m['kind'] == 'sube':
                g_ *= 1 + 0.9 * np.clip((t - m['t0']) / 0.3, 0, 1) * np.clip((m['t0'] + 3.0 - t) / 1.2, 0, 1)
            if m['kind'] == 'corte':
                g_ *= 1 - np.clip((t - (m['t0'] - 0.15)) / 0.15, 0, 1) * np.clip((m['t1'] + 0.1 - t) / 0.05, 0, 1)
    for e in SONG_EVENTS + SONG_MARKS:
        g_ *= 1 - np.clip((t - (e['t0'] - 0.6)) / 0.6, 0, 1) * np.clip((e['t0'] + e['dur'] + 1.0 - t) / 1.0, 0, 1)
    return g_

def read_audio(path, seconds, loop=False):
    args = ['ffmpeg', '-v', 'error'] + (['-stream_loop', '-1'] if loop else []) + ['-i', path, '-t', f"{seconds:.3f}",
            '-f', 'f32le', '-ac', '2', '-ar', str(SR), '-']
    a_ = np.frombuffer(subprocess.run(args, capture_output=True, check=True).stdout, np.float32).reshape(-1, 2)
    out = np.zeros((int(seconds * SR), 2), np.float32); out[:min(len(a_), len(out))] = a_[:len(out)]
    return out

def write_wav(path, arr):
    import wave
    with wave.open(path, 'wb') as wf:
        wf.setnchannels(2); wf.setsampwidth(2); wf.setframerate(SR)
        wf.writeframes((np.clip(arr, -1, 1) * 32767).astype(np.int16).tobytes())

if BEEPS:                                                    # "piiip": silencia la palabra y pone el tono
    vox = read_audio(VOICE_PROC, L)
    for a_, b_ in BEEPS:
        i0, i1 = int(max(0, a_ - 0.02) * SR), int(min(L, b_ + 0.02) * SR)
        if i1 <= i0: continue
        tt_ = np.arange(i1 - i0) / SR
        env_ = np.minimum(1, np.minimum(tt_ / 0.01, (tt_[-1] - tt_) / 0.01 + 1e-9))
        vox[i0:i1] = (np.sin(2 * np.pi * 1000 * tt_) * 0.28 * env_)[:, None]
    write_wav(VOICE_PROC, vox)
def trailer_music(dur):
    """Música épica de tráiler generada aquí (sin copyright): dron grave + pulsos que aceleran + subida final."""
    n = int(dur * SR); tt = np.arange(n) / SR; rng_ = np.random.default_rng(11)
    drone = sum(np.sin(2 * np.pi * f * tt + rng_.uniform(0, 6)) * a for f, a in ((55, .5), (55.4, .4), (82.4, .3), (110, .15), (164.8, .08)))
    drone *= 0.6 + 0.4 * np.sin(2 * np.pi * 0.07 * tt) ** 2
    pulse = np.zeros(n); bpm0, bpm1 = 70, 140; t_ = 0.0
    while t_ < dur - 0.3:
        i0 = int(t_ * SR); k_ = np.arange(min(int(0.25 * SR), n - i0)) / SR
        pulse[i0:i0 + len(k_)] += np.sin(2 * np.pi * 50 * k_) * np.exp(-k_ * 18) * (0.4 + 0.6 * t_ / dur)
        t_ += 60 / (bpm0 + (bpm1 - bpm0) * (t_ / dur) ** 2)
    rise = np.clip((tt - (dur - 2.5)) / 2.5, 0, 1) ** 2
    noise = np.convolve(rng_.standard_normal(n), np.ones(40) / 40, 'same') * rise * 0.6
    tone = np.sin(2 * np.pi * (300 + 900 * rise) * tt) * rise * 0.12
    x = (drone * 0.55 + pulse * 0.9 + noise + tone) * np.minimum(1, tt / 1.5)
    return np.stack([x, np.roll(x, 240)], 1).astype(np.float32) * 0.6
TRAILER_BED = None
if RUPT_F:
    rt = RUPT_F / FPS
    if MUSICA_TRAILER.strip().lower() != 'generada' and find_song(MUSICA_TRAILER):
        TRAILER_BED = read_audio(find_song(MUSICA_TRAILER), rt)
    else:
        TRAILER_BED = trailer_music(rt)
    TRAILER_BED[-int(0.06 * SR):] *= np.linspace(1, 0, int(0.06 * SR))[:, None]   # se corta en seco con el rayón
    if VOZ_TRAILER:                                          # voz de narrador: más grave y con eco de sala grande
        fxv = f"{WORK}/voz_trailer.wav"
        run(['ffmpeg', '-y', '-v', 'error', '-i', VOICE_PROC, '-af',
             "asetrate=48000*0.9,aresample=48000,atempo=1.1111,aecho=0.8:0.6:60|120:0.3|0.2,bass=g=4", '-ar', str(SR), fxv])
        dry, wet = read_audio(VOICE_PROC, L), read_audio(fxv, L)
        cut = int(rt * SR); xf = int(0.05 * SR)
        mixv = dry.copy(); mixv[:cut] = wet[:cut]
        mixv[cut - xf:cut] = wet[cut - xf:cut] * np.linspace(1, 0, xf)[:, None] + dry[cut - xf:cut] * np.linspace(0, 1, xf)[:, None]
        write_wav(VOICE_PROC, mixv)
BED_RAW = read_audio(BG_WAV, L, loop=True)
def build_audio(vol_musica):
    """Mezcla final: voz + música de fondo (con momentos clave y ducking) + efectos + canciones pedidas."""
    tt_ = np.arange(len(BED_RAW)) / SR
    bed = BED_RAW * (music_gain(tt_) * vol_musica)[:, None]
    if TRAILER_BED is not None:
        bed[:len(TRAILER_BED)] += TRAILER_BED[:len(bed)] * min(1.0, vol_musica * 4.5)
    fo = max(0, L - 2); bed[int(fo * SR):] *= np.linspace(1, 0, len(bed) - int(fo * SR))[:, None]
    BED_WAV = f"{WORK}/fondo_mezcla.wav"; write_wav(BED_WAV, bed)
    inputs = ['-i', VOICE_PROC, '-i', BED_WAV, '-i', SFX_WAV]
    fc = ["[0:a]asplit=2[v1][v2]", "[1:a]anull[m]",
          "[m][v2]sidechaincompress=threshold=0.03:ratio=6:attack=30:release=500[md]",
          "[2:a]aformat=channel_layouts=stereo[fx]"]
    mix_in = ['[v1]', '[md]', '[fx]']
    for j, e in enumerate(SONG_EVENTS):
        inputs += ['-i', e['file']]
        sdur = float(probe(e['file'])['duration'] or 60)
        ss = min(max(0.0, sdur * 0.3), max(0.0, sdur - e['dur']))
        fc.append(f"[{j + 3}:a]aresample={SR},aformat=channel_layouts=stereo,atrim={ss:.2f}:{ss + e['dur'] + 1:.2f},asetpts=N/SR/TB,"
                  f"volume={VOLUMEN_CANCIONES},afade=t=in:d=0.5:curve=qsin,afade=t=out:st={max(0, e['dur'] - 0.2):.2f}:d=1.2:curve=qsin,"
                  f"adelay={int((e['t0'] - 0.25) * 1000)}|{int((e['t0'] - 0.25) * 1000)}[s{j}]")
        mix_in.append(f'[s{j}]')
    fc.append(f"{''.join(mix_in)}amix=inputs={len(mix_in)}:normalize=0:duration=longest,alimiter=limit=0.95,apad,atrim=0:{L:.4f}[o]")
    run(['ffmpeg', '-y', '-v', 'error', *inputs, '-filter_complex', ';'.join(fc), '-map', '[o]', '-ar', str(SR), '-t', f"{L:.3f}", MIX_WAV])
MIX_WAV = f"{WORK}/mix.wav"
build_audio(VOLUMEN_MUSICA)
print("Audio listo:", MIX_WAV)

In [ ]:
#@title 7 · Look CINE: película (halación, grano, destellos), desenfoque IA, cámara lenta IA, barras, textos de tráiler + todo lo anterior
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import urllib.request
FONT_DIR = f"{WORK}/fonts"; os.makedirs(FONT_DIR, exist_ok=True)
GF = 'https://github.com/google/fonts/raw/main/'
FONT_URLS = {'Anton-Regular.ttf': GF + 'ofl/anton/Anton-Regular.ttf',
             'BebasNeue-Regular.ttf': GF + 'ofl/bebasneue/BebasNeue-Regular.ttf',
             'Cinzel.ttf': GF + 'ofl/cinzel/Cinzel%5Bwght%5D.ttf',
             'VT323-Regular.ttf': GF + 'ofl/vt323/VT323-Regular.ttf',
             'Bangers-Regular.ttf': GF + 'ofl/bangers/Bangers-Regular.ttf'}
for fn, url in FONT_URLS.items():
    p = os.path.join(FONT_DIR, fn)
    if not os.path.exists(p):
        try: urllib.request.urlretrieve(url, p)
        except Exception as e: print("No pude bajar", fn, e)
for f_ in USER_FONTS:                                   # tus fuentes de Server/fuentes
    shutil.copy(f_, FONT_DIR)
ROLE_FONT = {'sub': FUENTE_SUBTITULOS.strip(), 'card': FUENTE_TARJETAS.strip(), 'title': FUENTE_TITULOS.strip()}
for r_, fn_ in ROLE_FONT.items():
    if fn_ and not os.path.exists(os.path.join(FONT_DIR, fn_)):
        print(f"⚠ No encuentro la fuente «{fn_}» en {FONTS_DIR_USER}; uso la de por defecto."); ROLE_FONT[r_] = ''
def font(fn, size, *alts, role=None):
    first = (ROLE_FONT.get(role),) if role and ROLE_FONT.get(role) else ()
    for cand in [os.path.join(FONT_DIR, f) for f in first + (fn,) + alts] + ['/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf']:
        try:
            f = ImageFont.truetype(cand, size)
            if 'Cinzel' in cand:
                try: f.set_variation_by_axes([800])     # Cinzel en negrita
                except Exception: pass
            return f
        except Exception: pass
    return ImageFont.load_default(size=size)

# ---------- Color de cine cálido (verdes suaves) + grano + viñeta ----------
def curve(points):
    xs, ys = zip(*points); return np.clip(np.interp(np.arange(256), xs, ys), 0, 255).astype(np.uint8)
LB, LG, LR = (curve([(0, 14), (70, 70), (150, 140), (255, 228)]), curve([(0, 8), (128, 124), (255, 244)]),
              curve([(0, 6), (90, 92), (190, 200), (255, 255)]))
def grade(fr):
    if GRADE_MIX != 1.0: return cv2.addWeighted(_grade(fr), GRADE_MIX, fr, 1 - GRADE_MIX, 0)
    return _grade(fr)
def _grade(fr):
    b, g, r = cv2.split(fr); fr = cv2.merge([cv2.LUT(b, LB), cv2.LUT(g, LG), cv2.LUT(r, LR)])
    hsv = cv2.cvtColor(fr, cv2.COLOR_BGR2HSV).astype(np.int16)
    green = (hsv[..., 0] > 30) & (hsv[..., 0] < 85)
    hsv[..., 1] = np.where(green, hsv[..., 1] * 0.78, hsv[..., 1] * 0.97)
    return cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2BGR)
_vig = None; _grain = [np.random.normal(0, 5, (480, 270)).astype(np.float32) for _ in range(8)]; _gi = [0]
def finish(fr):
    global _vig
    if _vig is None:
        yy, xx = np.mgrid[0:OUT_H, 0:OUT_W]; d = np.sqrt(((xx - OUT_W / 2) / (OUT_W / 2)) ** 2 + ((yy - OUT_H / 2) / (OUT_H / 2)) ** 2)
        _vig = (1 - 0.28 * np.clip(d - 0.5, 0, 1) ** 1.4)[..., None].astype(np.float32)
    _gi[0] += 1; n = cv2.resize(_grain[_gi[0] % 8], (OUT_W, OUT_H))[..., None]
    return np.clip(fr.astype(np.float32) * _vig + n, 0, 255).astype(np.uint8)

def paste_rgba(frame, rgba, x0, y0, alpha=1.0):
    h, w = rgba.shape[:2]; H, W = frame.shape[:2]
    xa, ya, xb, yb = max(0, x0), max(0, y0), min(W, x0 + w), min(H, y0 + h)
    if xa >= xb or ya >= yb: return
    sub = rgba[ya - y0: yb - y0, xa - x0: xb - x0]
    a = sub[..., 3:4].astype(np.float32) / 255 * alpha
    frame[ya:yb, xa:xb] = (frame[ya:yb, xa:xb] * (1 - a) + sub[..., [2, 1, 0]] * a).astype(np.uint8)

# ---------- Persona: encuadre y máscara ----------
from ultralytics import YOLO
seg_model = YOLO('yolo11l-seg.pt')
def person_mask(fr):
    r = seg_model.predict(fr, classes=[0], conf=0.25, verbose=False, retina_masks=True, imgsz=640)[0]
    if r.masks is None or not len(r.masks): return np.zeros(fr.shape[:2], np.float32)
    m = r.masks.data.cpu().numpy().max(0).astype(np.float32)
    if m.shape != fr.shape[:2]: m = cv2.resize(m, (fr.shape[1], fr.shape[0]))
    return cv2.GaussianBlur(cv2.dilate(m, np.ones((7, 7), np.uint8)), (0, 0), 3)
def track_centers(pc):
    c = clips[pc['clip']]; n = pc['n']
    cap = cv2.VideoCapture(c['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, pc['src_f0'])
    xs, ys, hs, ts = [], [], [], []
    for i in range(n):
        ok, fr = cap.read()
        if not ok: break
        if i % 4 == 0:
            r = seg_model.predict(fr, classes=[0], conf=0.3, verbose=False, imgsz=640)[0]
            if len(r.boxes):
                b = r.boxes.xyxy.cpu().numpy(); a = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1]); j = int(a.argmax())
                xs.append(float((b[j, 0] + b[j, 2]) / 2 / fr.shape[1]))
                ys.append(float((b[j, 1] + 0.16 * (b[j, 3] - b[j, 1])) / fr.shape[0]))     # altura aprox. de la cara
                hs.append(float((b[j, 3] - b[j, 1]) / fr.shape[0]))                         # qué tan grande sale la persona
                ts.append(i)
    cap.release()
    if not xs: return np.full(n, 0.5), np.full(n, 0.3), np.full(n, 0.8)
    def smooth(v):
        raw = np.interp(np.arange(n), ts, v); out = np.empty(n); out[0] = np.median(raw[:15])
        for i in range(1, n): out[i] = out[i - 1] * 0.94 + raw[i] * 0.06
        return out
    return smooth(xs), smooth(ys), smooth(hs)
def crop_frame(fr, cxn, zoom, W=OUT_W, H=OUT_H, ytop=0.35, cyn=None):
    h, w = fr.shape[:2]; tgt = W / H
    cw, ch = (h * tgt, h) if w / h > tgt else (w, w / tgt)
    cw, ch = cw / zoom, ch / zoom
    x0 = np.clip(cxn * w - cw / 2, 0, w - cw)
    y0 = np.clip((h - ch) * ytop if cyn is None else cyn * h - ch * 0.4, 0, h - ch)
    M = np.float32([[W / cw, 0, -x0 * W / cw], [0, H / ch, -y0 * H / ch]])
    return cv2.warpAffine(fr, M, (W, H), flags=cv2.INTER_CUBIC)
for pc in PIECES:
    if pc['kind'] == 'cine': continue
    c = clips[pc['clip']]
    pc['cx'], pc['cy'], pc['ph'] = track_centers(pc)
    pc['fx'] = pc['cx'].copy()                               # centro real de la persona (para los zooms)
    if c['W'] / c['H'] <= OUT_W / OUT_H: pc['cx'] = np.full(pc['n'], 0.5)

# ---------- Subtítulos amarillos estilo davecast (palabra clave GRANDE, con brillo) ----------
YELLOW = (255, 196, 18)
CAP_Y, CAP_SCALE, GRADE_MIX = 0.70, 1.0, 1.0           # la revisión de IA puede ajustarlos
def glow_text_rgba(lines, color=YELLOW, glow=0.85, role='sub'):
    """lines = [(texto, tamaño)]; centrado, con brillo amarillo y sombra suave."""
    fonts = [font('Anton-Regular.ttf', sz, 'BebasNeue-Regular.ttf', role=role) for _, sz in lines]
    widths = [f.getlength(t) for (t, _), f in zip(lines, fonts)]
    W = int(max(widths)) + 120; H = sum(int(sz * 1.08) for _, sz in lines) + 120
    img = Image.new('RGBA', (W, H), (0, 0, 0, 0)); d = ImageDraw.Draw(img); y = 60
    for (t, sz), f, wd in zip(lines, fonts, widths):
        d.text(((W - wd) / 2, y), t, font=f, fill=color + (255,)); y += int(sz * 1.08)
    a = np.array(img)
    al = a[..., 3].astype(np.float32) / 255
    halo = cv2.GaussianBlur(al, (0, 0), 14) * glow
    shadow = cv2.GaussianBlur(al, (0, 0), 5) * 0.7
    out = np.zeros_like(a, dtype=np.float32)
    out[..., :3] = np.array(color, np.float32)
    comb = np.maximum(al, halo)
    rgb = out[..., :3] * (comb[..., None] > 0) * 1.0
    # sombra oscura debajo del brillo para que se lea sobre fondos claros
    base = np.dstack([np.zeros_like(al)] * 3 + [shadow])
    top = np.dstack([rgb, comb])
    a_out = top[..., 3] + base[..., 3] * (1 - top[..., 3])
    rgb_out = (top[..., :3] * top[..., 3:4]) / np.maximum(a_out[..., None], 1e-6)
    return np.dstack([rgb_out, a_out * 255]).clip(0, 255).astype(np.uint8)

_cap_cache = {}
def caption_rgba(grp):
    key = tuple(w[2] for w in grp) + tuple(w[3] for w in grp)
    if key in _cap_cache: return _cap_cache[key]
    clean = lambda s: re.sub(r'[{}\\]', '', s).upper().strip(' ,')
    big = [clean(w[2]) for w in grp if w[3]]; small = [clean(w[2]) for w in grp if not w[3]]
    if big:
        lines = ([(' '.join(small), 62)] if small else []) + [(' '.join(big), 132 if len(' '.join(big)) <= 10 else 104)]
    else:
        txt = ' '.join(clean(w[2]) for w in grp)
        lines = [(txt, 84 if len(txt) <= 16 else 66)]
    r = glow_text_rgba(lines); _cap_cache[key] = r
    return r
def draw_caption(frame, t):
    for grp in CAPS:
        if grp[0][0] <= t < grp[-1][1] + 0.12:
            r = caption_rgba(grp)
            if CAP_SCALE != 1.0: r = cv2.resize(r, None, fx=CAP_SCALE, fy=CAP_SCALE)
            h, w = r.shape[:2]
            pop = min(1.0, (t - grp[0][0]) / 0.12)
            if pop < 1:
                s_ = 0.86 + 0.14 * pop; r = cv2.resize(r, None, fx=s_, fy=s_); h, w = r.shape[:2]
            if w > OUT_W * 0.92:
                k_ = OUT_W * 0.92 / w; r = cv2.resize(r, None, fx=k_, fy=k_); h, w = r.shape[:2]
            paste_rgba(frame, r, (OUT_W - w) // 2, int(OUT_H * CAP_Y) - h // 2)
            return

# ---------- Pantalla negra con textura (notas del editor) ----------
def grunge_bg(seed=0):
    rng = np.random.default_rng(seed)
    base = cv2.GaussianBlur(rng.random((OUT_H // 8, OUT_W // 8)).astype(np.float32), (0, 0), 3)
    base = cv2.resize(base, (OUT_W, OUT_H)) * 22 + 8
    base += rng.normal(0, 4, (OUT_H, OUT_W))
    img = np.dstack([base] * 3).clip(0, 255).astype(np.uint8)
    for _ in range(40):                                  # rayones y polvo
        x, y = int(rng.integers(OUT_W)), int(rng.integers(OUT_H)); l = int(rng.integers(20, 160))
        ang = rng.uniform(0, np.pi)
        cv2.line(img, (x, y), (int(x + l * np.cos(ang)), int(y + l * np.sin(ang))), (60, 60, 60), 1, cv2.LINE_AA)
    return img
GRUNGE = [grunge_bg(i) for i in range(3)]
_card_cache = {}
def draw_card(frame, card, g):
    t = (g - card['f0']) / FPS; T = (card['f1'] - card['f0']) / FPS
    key = card['text']
    if key not in _card_cache:
        words = card['text'].split()
        first, rest = words[0], words[1:]
        lines = [(first, 210 if len(first) <= 8 else 150)]
        mid = (len(rest) + 1) // 2
        if rest: lines.append((' '.join(rest[:mid]), 96))
        if len(rest) > 1: lines.append((' '.join(rest[mid:]), 70))
        _card_cache[key] = glow_text_rgba(lines, glow=1.0, role='card')
    r = _card_cache[key]
    frame[:] = GRUNGE[hash(key) % 3]
    s_ = 1.0 + 0.05 * t / T
    shake = int(10 * max(0, 1 - t / 0.15)) * (1 if g % 2 else -1)
    rr = cv2.resize(r, None, fx=s_, fy=s_)
    if rr.shape[1] > OUT_W * 0.92: k_ = OUT_W * 0.92 / rr.shape[1]; rr = cv2.resize(rr, None, fx=k_, fy=k_)
    h, w = rr.shape[:2]
    flick = 0.85 + 0.15 * (np.random.rand() > 0.08)
    paste_rgba(frame, rr, (OUT_W - w) // 2 + shake, (OUT_H - h) // 2, flick * min(1, t / 0.08, (T - t) / 0.12))

# ---------- Texto 3D dorado ----------
def gold3d_rgba(text, size=150):
    f = font('Cinzel.ttf', size, 'BebasNeue-Regular.ttf', role='title')
    words = text.split(); lines, cur = [], ''
    for w in words:
        if cur and len(cur) + 1 + len(w) > 12: lines.append(cur); cur = w
        else: cur = (cur + ' ' + w).strip()
    lines.append(cur)
    depth = max(8, size // 12); pad = size // 2
    W = int(max(f.getlength(l) for l in lines)) + 2 * pad + depth; lh = int(size * 1.15); H = lh * len(lines) + 2 * pad + depth
    mask = Image.new('L', (W, H), 0); d = ImageDraw.Draw(mask)
    for k, l in enumerate(lines):
        d.text(((W - depth - f.getlength(l)) / 2, pad + k * lh), l, font=f, fill=255)
    m = np.array(mask).astype(np.float32) / 255
    out = np.zeros((H, W, 4), np.float32)
    for i in range(depth, 0, -1):                        # extrusión (cara lateral marrón-dorada)
        sh = np.roll(np.roll(m, i, 0), i, 1); c_ = 0.35 + 0.35 * (1 - i / depth)
        col = np.array([150, 95, 25]) * c_
        out[..., :3] = out[..., :3] * (1 - sh[..., None]) + col * sh[..., None]
        out[..., 3] = np.maximum(out[..., 3], sh)
    yy = np.linspace(0, 1, H)[:, None]
    grad = np.stack([255 - 30 * yy, 236 - 120 * yy, 160 - 140 * yy], -1) * np.ones((1, W, 1))   # oro: claro arriba, ámbar abajo
    bl = cv2.GaussianBlur(m, (0, 0), 2.2)
    bevel = np.clip(cv2.Sobel(bl, cv2.CV_32F, 0, 1, ksize=3) * -2.5 + cv2.Sobel(bl, cv2.CV_32F, 1, 0, ksize=3) * -1.5, -0.6, 0.9)[..., None]
    face = np.clip(grad * (1 + bevel), 0, 255)
    out[..., :3] = out[..., :3] * (1 - m[..., None]) + face * m[..., None]
    out[..., 3] = np.maximum(out[..., 3], m)
    halo = cv2.GaussianBlur(out[..., 3], (0, 0), size / 7) * 0.55   # brillo cálido
    a = np.maximum(out[..., 3], halo)
    rgb = np.where(out[..., 3:4] > 0.02, out[..., :3], np.array([255, 170, 60], np.float32))
    res = np.dstack([rgb, a * 255]).clip(0, 255).astype(np.uint8)
    ys, xs = np.nonzero(res[..., 3] > 10)                       # recorta el margen vacío
    return res[max(0, ys.min() - 6): ys.max() + 7, max(0, xs.min() - 6): xs.max() + 7] if len(xs) else res

def plane_quad(w, h, cx, cy, yaw, pitch, scale, focal=1600):
    pts = np.array([[-w / 2, -h / 2, 0], [w / 2, -h / 2, 0], [w / 2, h / 2, 0], [-w / 2, h / 2, 0]], np.float32) * scale
    cy_, sy = math.cos(math.radians(yaw)), math.sin(math.radians(yaw))
    cp, sp = math.cos(math.radians(pitch)), math.sin(math.radians(pitch))
    R = np.array([[cy_, 0, sy], [0, 1, 0], [-sy, 0, cy_]]) @ np.array([[1, 0, 0], [0, cp, -sp], [0, sp, cp]])
    P = pts @ R.T; z = P[:, 2] + focal
    return np.stack([cx + P[:, 0] * focal / z, cy + P[:, 1] * focal / z], 1).astype(np.float32)
def fit_quad(q):
    x0, x1 = q[:, 0].min(), q[:, 0].max(); lo, hi = OUT_W * 0.05, OUT_W * 0.95
    if x1 - x0 > hi - lo:
        c = q.mean(0); q = (q - c) * ((hi - lo) / (x1 - x0)) + c; x0, x1 = q[:, 0].min(), q[:, 0].max()
    q = q.copy(); q[:, 0] += max(0, lo - x0) - max(0, x1 - hi); q[:, 1] += max(0, OUT_H * 0.07 - q[:, 1].min())
    return q

class SceneText:
    """Título 3D dorado colocado en el fondo, detrás de ti, siguiendo la cámara (como 'STAY CREATIVE' en el cielo)."""
    def __init__(self, spec, frame, mask):
        self.rgba = gold3d_rgba(spec['text']); h, w = self.rgba.shape[:2]
        bg = 1 - mask; scale = min(0.86 * OUT_W / w, 0.2 * OUT_H / h); best = None
        for cy in (0.13, 0.2, 0.28):
            q = fit_quad(plane_quad(w, h, OUT_W / 2, OUT_H * cy, random.choice([-14, 14]), -8, scale))
            poly = np.zeros((OUT_H // 4, OUT_W // 4), np.uint8); cv2.fillConvexPoly(poly, (q / 4).astype(np.int32), 1)
            vis = bg[::4, ::4][poly > 0].mean() if poly.any() else 0
            if best is None or vis > best[0]: best = (vis, q)
        self.vis, self.quad0 = best
        self.behind = self.vis >= 0.55 and mask.mean() > 0.03   # si casi no se ve el fondo, va delante
        self.H = np.eye(3); self.prev = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        self.pts = cv2.goodFeaturesToTrack(self.prev, 300, 0.01, 12, mask=((mask < 0.5) * 255).astype(np.uint8))
    def update(self, frame):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if self.pts is not None and len(self.pts) >= 8:
            nxt, st, _ = cv2.calcOpticalFlowPyrLK(self.prev, gray, self.pts, None, winSize=(21, 21), maxLevel=3)
            g0, g1 = self.pts[st[:, 0] == 1], nxt[st[:, 0] == 1]
            if len(g0) >= 8:
                Hs, _ = cv2.findHomography(g0, g1, cv2.RANSAC, 3.0)
                if Hs is not None and abs(np.linalg.det(Hs[:2, :2]) - 1) < 0.2:
                    Hn = Hs @ self.H; c0 = self.quad0.mean(0); c1 = cv2.perspectiveTransform(c0.reshape(1, 1, 2), Hn)[0, 0]
                    if np.abs(c1 - c0).max() < OUT_W * 0.2: self.H = Hn
                self.pts = g1.reshape(-1, 1, 2)
        if self.pts is None or len(self.pts) < 60: self.pts = cv2.goodFeaturesToTrack(gray, 300, 0.01, 12)
        self.prev = gray
    def draw(self, frame, mask, t, T):
        q = fit_quad(cv2.perspectiveTransform(self.quad0.reshape(-1, 1, 2), self.H).reshape(4, 2))
        c = q.mean(0); rise = (1 - min(1, t / 0.5)) ** 2; q = (q - c) * (0.85 + 0.15 * (1 - rise)) + c + np.array([0, 40 * rise])
        h, w = self.rgba.shape[:2]
        M = cv2.getPerspectiveTransform(np.float32([[0, 0], [w, 0], [w, h], [0, h]]), q.astype(np.float32))
        wp = cv2.warpPerspective(self.rgba, M, (OUT_W, OUT_H), flags=cv2.INTER_LINEAR, borderValue=(0, 0, 0, 0))
        a = wp[..., 3:4].astype(np.float32) / 255 * min(1, t / 0.35, (T - t) / 0.3)
        if self.behind: a *= (1 - mask)[..., None]
        frame[:] = (frame * (1 - a) + wp[..., [2, 1, 0]] * a).astype(np.uint8)

# ---------- B-roll: pantalla completa, polaroid y TV vieja ----------
def blur_fill(img):
    bg = crop_frame(img, 0.5, 1.0); bg = cv2.GaussianBlur(cv2.resize(bg, (OUT_W // 6, OUT_H // 6)), (0, 0), 6)
    return (cv2.resize(bg, (OUT_W, OUT_H)) * 0.55).astype(np.uint8)
def fit_into(img, W, H):
    h, w = img.shape[:2]; k = min(W / w, H / h); return cv2.resize(img, (int(w * k), int(h * k)), interpolation=cv2.INTER_AREA)
class BrollPlayer:
    def __init__(self, ev):
        self.ev = ev; b = ev['b']; self.cap = None
        if b['kind'] == 'video':
            self.cap = cv2.VideoCapture(b['proxy']); n = b.get('nframes', 0)
            span = ev['f1'] - ev['f0']
            if n > span + 10: self.cap.set(cv2.CAP_PROP_POS_FRAMES, random.randint(0, n - span - 1))
        self.last = b.get('img'); self.rot = random.uniform(-4, 4); self.counter = random.randint(100, 900)
    def frame(self):
        if self.cap is not None:
            ok, fr = self.cap.read()
            if ok: self.last = fr
        return self.last if self.last is not None else np.zeros((720, 1280, 3), np.uint8)
    def draw(self, out, g):
        src = self.frame(); t = (g - self.ev['f0']) / FPS; T = (self.ev['f1'] - self.ev['f0']) / FPS
        style = self.ev['style']; z = 1 + 0.04 * t / T
        out[:] = blur_fill(src)
        if style == 'pantalla':
            if src.shape[1] / src.shape[0] < 0.75: out[:] = crop_frame(src, 0.5, z)
            else:
                im = fit_into(src, int(OUT_W * 0.96 * z), int(OUT_H * 0.6)); h, w = im.shape[:2]
                out[(OUT_H - h) // 2:(OUT_H - h) // 2 + h, (OUT_W - w) // 2:(OUT_W - w) // 2 + w] = im
        elif style == 'polaroid':
            out[:] = (out * 0.35 + GRUNGE[1] * 0.65).astype(np.uint8)
            im = fit_into(src, int(OUT_W * 0.74), int(OUT_H * 0.46)); h, w = im.shape[:2]
            m_, bot = 26, 120
            card = np.full((h + m_ + bot, w + 2 * m_, 4), (238, 236, 230, 255), np.uint8)
            card[m_:m_ + h, m_:m_ + w, :3] = im[..., ::-1]
            ang = self.rot + 1.5 * math.sin(t * 1.3); s_ = 0.92 + 0.08 * min(1, t / 0.3)
            ch, cw = card.shape[:2]; M = cv2.getRotationMatrix2D((cw / 2, ch / 2), ang, s_)
            cos, sin = abs(M[0, 0]), abs(M[0, 1]); nw, nh = int(ch * sin + cw * cos), int(ch * cos + cw * sin)
            M[0, 2] += nw / 2 - cw / 2; M[1, 2] += nh / 2 - ch / 2
            rc = cv2.warpAffine(card, M, (nw, nh), borderValue=(0, 0, 0, 0))
            sh = rc.copy(); sh[..., :3] = 0; sh[..., 3] = cv2.GaussianBlur(sh[..., 3], (0, 0), 18) // 2
            x0, y0 = (OUT_W - nw) // 2, int(OUT_H * 0.42) - nh // 2
            paste_rgba(out, sh, x0 + 18, y0 + 26); paste_rgba(out, rc, x0, y0)
        else:   # tv vieja
            sw, sh_ = int(OUT_W * 0.86), int(OUT_W * 0.86 * 0.78)
            scr = crop_frame(src, 0.5, z, sw, sh_)
            scr = cv2.addWeighted(scr, 0.9, cv2.GaussianBlur(scr, (0, 0), 2), 0.25, 0)
            scr[::4] = (scr[::4] * 0.72).astype(np.uint8)                           # líneas de barrido
            yy, xx = np.mgrid[0:sh_, 0:sw]; v = 1 - 0.45 * (((xx - sw / 2) / (sw / 2)) ** 2 + ((yy - sh_ / 2) / (sh_ / 2)) ** 2) ** 1.5
            scr = (scr * np.clip(v, 0, 1)[..., None]).astype(np.uint8)
            b_, g_, r_ = cv2.split(scr); scr = cv2.merge([np.roll(b_, -2, 1), g_, np.roll(r_, 2, 1)])
            f = font('VT323-Regular.ttf', 54); pil = Image.fromarray(scr[..., ::-1]); d = ImageDraw.Draw(pil)
            d.text((sw / 2 - 60, sh_ - 70), f"{self.counter + g - self.ev['f0']:06d}", font=f, fill=(225, 225, 225))
            scr = np.array(pil)[..., ::-1]
            out[:] = (out * 0.5).astype(np.uint8)
            x0, y0 = (OUT_W - sw) // 2, (OUT_H - sh_) // 2 - 80
            cv2.rectangle(out, (x0 - 40, y0 - 40), (x0 + sw + 40, y0 + sh_ + 40), (22, 22, 24), -1, cv2.LINE_AA)
            cv2.rectangle(out, (x0 - 40, y0 - 40), (x0 + sw + 40, y0 + sh_ + 40), (55, 55, 58), 4, cv2.LINE_AA)
            mask_ = np.zeros((sh_, sw), np.uint8); cv2.rectangle(mask_, (0, 0), (sw - 1, sh_ - 1), 255, -1)
            rr = 46; mask_[:rr, :rr] = 0; mask_[:rr, -rr:] = 0; mask_[-rr:, :rr] = 0; mask_[-rr:, -rr:] = 0
            for cx_, cy_ in ((rr, rr), (sw - rr, rr), (rr, sh_ - rr), (sw - rr, sh_ - rr)): cv2.circle(mask_, (cx_, cy_), rr, 255, -1)
            a = (mask_ / 255.0)[..., None]
            out[y0:y0 + sh_, x0:x0 + sw] = (out[y0:y0 + sh_, x0:x0 + sw] * (1 - a) + scr * a).astype(np.uint8)
            if random.random() < 0.08: out[y0:y0 + sh_, x0:x0 + sw] = cv2.add(out[y0:y0 + sh_, x0:x0 + sw], 25)   # parpadeo

# ---------- Trazos de luz neón ----------
class Neon:
    def __init__(self, ev):
        rng = np.random.default_rng(ev['f0']); n = 7
        pts = np.stack([np.linspace(-0.05, 1.05, n) * OUT_W, (0.35 + 0.25 * rng.random(n)) * OUT_H], 1)
        pts[:, 1] += np.sin(np.arange(n) * 2.1) * OUT_H * 0.05
        tt = np.linspace(0, 1, 400); idx = tt * (n - 1); i0 = np.clip(idx.astype(int), 0, n - 2); fr_ = idx - i0
        base = pts[i0] * (1 - fr_[:, None]) + pts[i0 + 1] * fr_[:, None]
        wig = np.stack([np.sin(tt * 38) * 55, np.cos(tt * 31) * 70], 1) * np.sin(tt * np.pi)[:, None]   # garabato
        self.path = (base + wig).astype(np.int32)
        self.col = [(255, 140, 40), (255, 60, 200), (60, 220, 255)][int(rng.integers(3))]
        self.ev = ev
    def draw(self, frame, g):
        t = (g - self.ev['f0']) / (self.ev['f1'] - self.ev['f0'])
        head = int(len(self.path) * min(1, t * 1.6)); tail = int(len(self.path) * max(0, t * 1.6 - 0.7))
        if head - tail < 2: return
        layer = np.zeros_like(frame); seg = self.path[tail:head].reshape(-1, 1, 2)
        cv2.polylines(layer, [seg], False, self.col[::-1], 34, cv2.LINE_AA)
        layer = cv2.GaussianBlur(layer, (0, 0), 16)
        cv2.polylines(layer, [seg], False, self.col[::-1], 12, cv2.LINE_AA)
        cv2.polylines(layer, [seg], False, (255, 255, 255), 5, cv2.LINE_AA)
        frame[:] = cv2.add(frame, layer)

# ---------- Spider-Verse: cómic (halftone, tinta, colores desfasados, animación a saltos) ----------
yy_, xx_ = np.mgrid[0:OUT_H, 0:OUT_W].astype(np.float32)
_ang = np.radians(18)
_u = (xx_ * np.cos(_ang) + yy_ * np.sin(_ang)) / 11; _v = (-xx_ * np.sin(_ang) + yy_ * np.cos(_ang)) / 11
DOTS = (np.sqrt((_u - np.round(_u)) ** 2 + (_v - np.round(_v)) ** 2) * 2).astype(np.float32)   # 0 en el centro del punto
def spiderverse(frame, t):
    f = frame.astype(np.float32)
    lum = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255
    f = np.round(f / 48) * 48                                                    # colores planos
    dots = (DOTS < (1 - lum) * 1.25).astype(np.float32)                           # puntos de imprenta en las sombras
    f *= (1 - 0.35 * dots)[..., None]
    b_, g_, r_ = cv2.split(f); off = 7 + int(3 * math.sin(t * 20))
    f = cv2.merge([np.roll(b_, -off, 1), g_, np.roll(np.roll(r_, off, 1), off // 2, 0)])   # desfase de impresión
    edges = cv2.Canny(cv2.GaussianBlur(frame, (5, 5), 0), 60, 140)
    edges = cv2.dilate(edges, np.ones((3, 3), np.uint8)).astype(np.float32)[..., None] / 255
    f = f * (1 - 0.85 * edges)                                                    # líneas de tinta
    return np.clip(f, 0, 255).astype(np.uint8)

# ====================== MODO CINE ======================
# ---------- Look de película: halación, destellos anamórficos, grano y vibración de proyector ----------
_film_g = [np.random.normal(0, 1, (OUT_H // 2, OUT_W // 2)).astype(np.float32) for _ in range(6)]
def film_look(fr, g):
    if not LOOK_PELICULA: return fr
    f = fr.astype(np.float32)
    small = cv2.resize(f, (OUT_W // 4, OUT_H // 4), interpolation=cv2.INTER_AREA)
    lum = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY) / 255
    hi = np.clip((lum - 0.72) / 0.28, 0, 1)
    halo = cv2.GaussianBlur(hi, (0, 0), 7)                                       # halación: brillo rojizo alrededor de las luces
    streak = cv2.blur(np.clip((lum - 0.9) / 0.1, 0, 1), (151, 1))                 # destello anamórfico horizontal
    halo = cv2.resize(halo, (OUT_W, OUT_H))[..., None]; streak = cv2.resize(streak, (OUT_W, OUT_H))[..., None]
    f = f + halo * np.array([20, 45, 110], np.float32) + streak * np.array([140, 70, 30], np.float32)
    lumf = cv2.cvtColor(np.clip(f, 0, 255).astype(np.uint8), cv2.COLOR_BGR2GRAY).astype(np.float32) / 255
    gr = cv2.resize(_film_g[g % 6], (OUT_W, OUT_H))
    f += (gr * (9 * (1 - np.abs(lumf - 0.5) * 1.4)))[..., None]                 # grano de película (más en los medios tonos)
    dx, dy = 0.6 * math.sin(g * 0.9), 0.5 * math.sin(g * 1.7 + 1)                  # vibración de proyector (gate weave)
    f = cv2.warpAffine(f, np.float32([[1, 0, dx], [0, 1, dy]]), (OUT_W, OUT_H), borderMode=cv2.BORDER_REFLECT)
    return np.clip(f, 0, 255).astype(np.uint8)

# ---------- Desenfoque de fondo con IA (profundidad Depth Anything + máscara de persona) ----------
DEPTH = None
if DESENFOQUE_IA:
    try:
        from transformers import pipeline as _hf_pipeline
        import torch as _torch
        DEPTH = _hf_pipeline('depth-estimation', model='depth-anything/Depth-Anything-V2-Small-hf',
                             device=0 if _torch.cuda.is_available() else -1)
        print("Desenfoque IA: Depth Anything V2 ✓")
    except Exception as e:
        print("Desenfoque IA sin modelo de profundidad (uso solo la máscara de persona):", str(e)[:150])
def depth_blur(fr, mask=None, strength=1.0, force=False):
    if not (DESENFOQUE_IA or force): return fr
    h, w = fr.shape[:2]
    if mask is None: mask = person_mask(fr)
    if (mask > 0.5).mean() < 0.02 and not force: return fr      # sin persona en cuadro: no se desenfoca nada
    weight = None
    if DEPTH is not None:
        try:
            d = np.array(DEPTH(Image.fromarray(cv2.cvtColor(cv2.resize(fr, (w // 3, h // 3)), cv2.COLOR_BGR2RGB)))['depth'], np.float32)
            d = cv2.resize(d, (w, h)); d = (d - d.min()) / (np.ptp(d) + 1e-6)          # 1 = cerca, 0 = lejos
            subj = np.median(d[mask > 0.5]) if (mask > 0.5).any() else np.percentile(d, 85)
            weight = np.clip((subj - d) / max(0.25, subj * 0.6), 0, 1)
        except Exception:
            weight = None
    if weight is None: weight = 1 - cv2.GaussianBlur(cv2.dilate(mask, np.ones((25, 25), np.uint8)), (0, 0), 12)
    weight = np.clip(weight * (1 - mask), 0, 1)[..., None] * strength
    small = cv2.resize(fr, (w // 3, h // 3), interpolation=cv2.INTER_AREA)
    bl = cv2.resize(cv2.GaussianBlur(small, (0, 0), 4.5), (w, h))
    return (fr * (1 - weight) + bl * weight).astype(np.uint8)

# ---------- Cámara lenta con IA (flujo óptico RAFT; si no está, flujo de OpenCV) ----------
RAFT = None
if CAMARA_LENTA_IA:
    try:
        import torch as _torch
        from torchvision.models.optical_flow import raft_small, Raft_Small_Weights
        _dev = 'cuda' if _torch.cuda.is_available() else 'cpu'
        RAFT = raft_small(weights=Raft_Small_Weights.DEFAULT).eval().to(_dev)
        print("Cámara lenta IA: RAFT ✓")
    except Exception as e:
        print("Cámara lenta sin RAFT (uso flujo óptico de OpenCV):", str(e)[:150])
def optical_flow(a, b):
    if RAFT is not None:
        import torch as _torch
        H_, W_ = (a.shape[0] // 8) * 8, (a.shape[1] // 8) * 8
        ta = _torch.from_numpy(cv2.resize(a, (W_, H_))[..., ::-1].copy()).permute(2, 0, 1)[None].float().to(_dev) / 127.5 - 1
        tb = _torch.from_numpy(cv2.resize(b, (W_, H_))[..., ::-1].copy()).permute(2, 0, 1)[None].float().to(_dev) / 127.5 - 1
        with _torch.no_grad(): fl = RAFT(ta, tb)[-1][0].permute(1, 2, 0).cpu().numpy()
        fl = cv2.resize(fl, (a.shape[1], a.shape[0])); fl[..., 0] *= a.shape[1] / W_; fl[..., 1] *= a.shape[0] / H_
        return fl
    ga, gb = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY), cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)
    return cv2.calcOpticalFlowFarneback(ga, gb, None, 0.5, 4, 25, 3, 7, 1.5, 0)
def interpolate(a, b, t):
    """Cuadro intermedio entre a y b (t en 0..1) deformando ambos con el flujo óptico."""
    h, w = a.shape[:2]; gx, gy = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    f01 = optical_flow(a, b); f10 = optical_flow(b, a)
    wa = cv2.remap(a, gx - t * f01[..., 0], gy - t * f01[..., 1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    wb = cv2.remap(b, gx - (1 - t) * f10[..., 0], gy - (1 - t) * f10[..., 1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    return cv2.addWeighted(wa, 1 - t, wb, t, 0)

class CinePlayer:
    """Da los cuadros de una toma de cine; en cámara lenta inventa los intermedios."""
    def __init__(self, pc):
        self.pc = pc; cc = pc['cc']; self.cap = cv2.VideoCapture(cc['proxy'])
        self.cap.set(cv2.CAP_PROP_POS_FRAMES, pc['src_f0'])
        self.step = (cc['fps'] / FPS) / (2.5 if pc['slow'] else 1)   # cuadros de origen por cuadro de salida
        self.pos = 0.0; self.buf = {}; self.next_idx = 0
    def _get(self, k):
        while self.next_idx <= k:
            ok, fr = self.cap.read()
            self.buf[self.next_idx] = fr if ok else self.buf.get(self.next_idx - 1)
            self.next_idx += 1
        for old in [i for i in self.buf if i < k - 2]: del self.buf[old]
        return self.buf[k]
    def frame(self, i):
        p = i * self.step; k = int(p); frac = p - k
        a = self._get(k)
        if frac < 0.08: return a
        b = self._get(k + 1)
        return a if b is None else interpolate(a, b, frac)

# ---------- Barras de cine en el tráiler (se van con el rayón) ----------
BAR_H = 300
def cine_bars(f, g):
    if not (BARRAS_CINE_EN_TRAILER and RUPT_F): return f
    if g < RUPT_F: k = 1.0
    elif g < RUPT_F + 8: k = 1 - (g - RUPT_F) / 8
    elif CIERRE_F and g >= CIERRE_F: k = min(1.0, (g - CIERRE_F) / 10)        # vuelven en el cierre épico
    else: return f
    hb = int(BAR_H * k)
    if hb > 0: f[:hb] = 0; f[OUT_H - hb:] = 0
    return f

# ---------- Texto de tráiler (blanco, serif, letras separadas, aparece lento) ----------
_tt_cache = {}
def draw_trailer_text(f, ev, g):
    t = (g - ev['f0']) / FPS; T = (ev['f1'] - ev['f0']) / FPS
    if ev['text'] not in _tt_cache:
        fnt = font('Cinzel.ttf', 92, 'BebasNeue-Regular.ttf', role='title')
        words = ev['text'].split(); lines, cur = [], ''
        for w_ in words:
            if cur and len(cur) + 1 + len(w_) > 16: lines.append(cur); cur = w_
            else: cur = (cur + ' ' + w_).strip()
        lines.append(cur)
        track = 10; widths = [sum(fnt.getlength(ch) + track for ch in l) for l in lines]
        W_ = int(max(widths)) + 80; H_ = len(lines) * 120 + 60
        img = Image.new('RGBA', (W_, H_), (0, 0, 0, 0)); d = ImageDraw.Draw(img)
        for k_, (l, wd) in enumerate(zip(lines, widths)):
            x = (W_ - wd) / 2
            for ch in l: d.text((x, 30 + k_ * 120), ch, font=fnt, fill=(245, 240, 230, 255)); x += fnt.getlength(ch) + track
        a = np.array(img); glow = cv2.GaussianBlur(a[..., 3], (0, 0), 10)
        a[..., 3] = np.maximum(a[..., 3], (glow * 0.45).astype(np.uint8))
        if a.shape[1] > OUT_W * 0.9: a = cv2.resize(a, None, fx=OUT_W * 0.9 / a.shape[1], fy=OUT_W * 0.9 / a.shape[1])
        _tt_cache[ev['text']] = a
    r = _tt_cache[ev['text']]
    s_ = 1.0 + 0.04 * t / max(T, 0.1); rr = cv2.resize(r, None, fx=s_, fy=s_)
    al = min(1, t / 0.6, (T - t) / 0.4)
    paste_rgba(f, rr, (OUT_W - rr.shape[1]) // 2, (OUT_H - rr.shape[0]) // 2, max(0, al))

# ---------- Efectos que pides por voz y transiciones ----------
def fx_bn(f):
    g_ = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
    g_ = cv2.convertScaleAbs(g_, alpha=1.25, beta=-25)               # blanco y negro con contraste dramático
    return cv2.cvtColor(g_, cv2.COLOR_GRAY2BGR)
def fx_temblor(f, k):
    amp = 38 * max(0.15, 1 - k / 18); rng_ = np.random.default_rng(k)
    dx, dy, ang = rng_.uniform(-amp, amp), rng_.uniform(-amp, amp), rng_.uniform(-2, 2) * amp / 38
    M = cv2.getRotationMatrix2D((OUT_W / 2, OUT_H / 2), ang, 1.06); M[0, 2] += dx; M[1, 2] += dy
    return cv2.warpAffine(f, M, (OUT_W, OUT_H), borderMode=cv2.BORDER_REFLECT)
def fx_glitch(f, k):
    rng_ = np.random.default_rng(1000 + k); out = f.copy()
    b_, g_, r_ = cv2.split(out); off = int(rng_.integers(8, 30))
    out = cv2.merge([np.roll(b_, -off, 1), g_, np.roll(r_, off, 1)])
    for _ in range(6):                                                # franjas desplazadas
        y0 = int(rng_.integers(0, OUT_H - 80)); hh = int(rng_.integers(20, 120)); out[y0:y0 + hh] = np.roll(out[y0:y0 + hh], int(rng_.integers(-120, 120)), 1)
    return out
def apply_transition(f, g):
    """Efecto alrededor de un corte que pediste ('Editor, transición …')."""
    for tr in TRANS:
        d = g - tr['f']; k = tr['kind']
        if k == 'flash' and -1 <= d < 7:
            w_ = max(0.0, 1 - (d + 1) / 8); f = cv2.addWeighted(f, 1 - w_, np.full_like(f, 255), w_, 0)
        elif k == 'glitch' and -3 <= d < 5: f = fx_glitch(f, g)
        elif k == 'barrido' and -4 <= d < 4:
            a_ = 1 - abs(d + 0.5) / 4.5; shift = int((1 if d < 0 else -1) * a_ * OUT_W * 0.35)
            f = cv2.blur(np.roll(f, shift, 1), (max(1, int(121 * a_)) | 1, 1))
        elif k == 'zoom' and -5 <= d < 5:
            a_ = 1 - abs(d + 0.5) / 5.5; s_ = 1 + 0.35 * a_
            M = cv2.getRotationMatrix2D((OUT_W / 2, OUT_H / 2), 0, s_); z_ = cv2.warpAffine(f, M, (OUT_W, OUT_H), borderMode=cv2.BORDER_REFLECT)
            f = cv2.addWeighted(z_, 0.6, cv2.blur(z_, (1, max(1, int(61 * a_)) | 1)), 0.4, 0)
        elif k == 'negro' and -12 <= d < 12:
            f = (f * (abs(d + 0.5) / 12.5)).astype(np.uint8)
        elif k == 'rayon' and -2 <= d < 4:
            f = fx_temblor(fx_bn(f) if d < 0 else f, g)
    return f
print("Look listo.")

In [ ]:
#@title 8 · Revisión de diseño con IA: mira fotogramas de muestra, opina (con sarcasmo) y ajusta antes del render final
def active(lst, g): return next((x for x in lst if x['f0'] <= g < x['f1']), None)

def compose(g, pc, i, fr, st):
    """Arma el fotograma g de salida. st = estado (seguimiento de títulos, b-roll, neón, Spider-Verse)."""
    tt = g / FPS; n = pc['n']
    card, br, fx, zm = active(CARDS, g), active(BROLLS, g), active(FX, g), active(ZOOMS, g)
    if pc['kind'] == 'cine':                                 # toma de cine (cámara lenta si toca) con empuje lento
        pl = st.setdefault('cine', {})
        if id(pc) not in pl: pl.clear(); pl[id(pc)] = CinePlayer(pc)
        src = pl[id(pc)].frame(i)
        if src is None: src = np.zeros((OUT_H, OUT_W, 3), np.uint8)
        f = grade(crop_frame(src, 0.5, 1.0 + 0.06 * i / max(1, n)))
        f = depth_blur(f, strength=0.7)
        card = None
    elif card is not None:
        f = np.empty((OUT_H, OUT_W, 3), np.uint8); draw_card(f, card, g)
    elif br is not None:
        f = np.empty((OUT_H, OUT_W, 3), np.uint8)
        st.setdefault('broll', {}).setdefault(id(br), BrollPlayer(br)).draw(f, g); f = grade(f)
    else:
        seg = max(j for j, m_ in enumerate(pc['zmarks']) if m_ <= i)
        s_start = pc['zmarks'][seg]; s_end = pc['zmarks'][seg + 1] if seg + 1 < len(pc['zmarks']) else n
        z = pc['zlevels'][seg] * (1 + 0.03 * (i - s_start) / max(1, s_end - s_start))
        cx = float(np.clip(pc['cx'][min(i, len(pc['cx']) - 1)] + pc['xoff'][seg], 0, 1)); cy = None
        if zm is not None:                                   # zoom de risa: de golpe a la cara, con sacudida
            k_ = g - zm['f0']; snap = min(1.0, k_ / 3); j_ = min(i, len(pc['cy']) - 1)
            ph = float(pc['ph'][j_]); c_ = clips[pc['clip']]
            vert = c_['W'] / c_['H'] <= OUT_W / OUT_H
            z_face = float(np.clip((1.1 if vert else 0.62) / max(ph, 0.1), 1.6, 3.4))   # que la cara llene la pantalla
            z = z + (z_face - z) * snap
            cx = float(pc['fx'][j_]); cy = float(pc['cy'][j_])
            if k_ < 8: cx += (0.012 if k_ % 2 else -0.012) * (1 - k_ / 8)
        f = grade(crop_frame(fr, cx, z, cyn=cy))
        if fx is not None and fx['kind'] == 'desenfoca': f = depth_blur(f, strength=1.0, force=True)
        elif DESENFOQUE_IA and fx is None: f = depth_blur(f)          # fondo desenfocado como lente de cine
        if fx is not None and fx['kind'] == 'spiderverse':
            if (g - fx['f0']) % 2 == 1 and st.get('sv') is not None: f = st['sv'].copy()
            else: f = spiderverse(f, tt); st['sv'] = f.copy()
        ttl = active(TITLES, g)
        if ttl is not None and zm is None:
            mask = person_mask(f); sc = st.setdefault('scene', {})
            if id(ttl) not in sc: sc[id(ttl)] = SceneText(ttl, f, mask)
            else: sc[id(ttl)].update(f)
            sc[id(ttl)].draw(f, mask, (g - ttl['f0']) / FPS, (ttl['f1'] - ttl['f0']) / FPS)
    if fx is not None and fx['kind'] == 'neon' and card is None:
        st.setdefault('neon', {}).setdefault(id(fx), Neon(fx)).draw(f, g)
    if fx is not None and card is None:                      # efectos que pediste por voz
        if fx['kind'] == 'bn': f = fx_bn(f)
        elif fx['kind'] == 'temblor': f = fx_temblor(f, g - fx['f0'])
        elif fx['kind'] == 'tv':
            tvp = st.setdefault('tv', {}).get(id(fx))
            if tvp is None:
                tvp = BrollPlayer({'b': {'kind': 'image', 'img': f.copy(), 'name': 'tv'}, 'style': 'tv', 'f0': fx['f0'], 'f1': fx['f1']})
                st['tv'][id(fx)] = tvp
            tvp.last = f.copy(); out_ = np.empty_like(f); tvp.draw(out_, g); f = out_
    if TRANS: f = apply_transition(f, g)
    if CTA['f0'] <= g < CTA['f1'] and TITULOS_3D_DORADOS:
        if st.get('cta') is None:
            r_ = gold3d_rgba(CTA['text'], 120)
            if r_.shape[1] > OUT_W * 0.9: r_ = cv2.resize(r_, None, fx=OUT_W * 0.9 / r_.shape[1], fy=OUT_W * 0.9 / r_.shape[1])
            st['cta'] = r_
        paste_rgba(f, st['cta'], (OUT_W - st['cta'].shape[1]) // 2, int(OUT_H * 0.12), min(1, (g - CTA['f0']) / FPS / 0.4))
    tx = active(TRAILER_TXT, g)
    if tx is not None: draw_trailer_text(f, tx, g)
    if card is None:
        if RUPT_F and g < RUPT_F: draw_caption_trailer(f, tt)
        else: draw_caption(f, tt)
    f = finish(film_look(f, g))
    return cine_bars(f, g)

def draw_caption_trailer(frame, t):
    """Subtítulo del narrador en el tráiler: blanco, serif, discreto, encima de la barra de abajo."""
    for grp in CAPS:
        if grp[0][0] <= t < grp[-1][1] + 0.12:
            key = ('tr',) + tuple(w[2] for w in grp)
            if key not in _cap_cache:
                fnt = font('Cinzel.ttf', 50, 'BebasNeue-Regular.ttf', role='title'); txt = ' '.join(w[2] for w in grp).upper()
                W_ = int(fnt.getlength(txt)) + 40; img = Image.new('RGBA', (W_, 90), (0, 0, 0, 0))
                ImageDraw.Draw(img).text((20, 15), txt, font=fnt, fill=(240, 236, 228, 255))
                a = np.array(img)
                if a.shape[1] > OUT_W * 0.9: a = cv2.resize(a, None, fx=OUT_W * 0.9 / a.shape[1], fy=OUT_W * 0.9 / a.shape[1])
                _cap_cache[key] = a
            r = _cap_cache[key]; y = OUT_H - (BAR_H if (BARRAS_CINE_EN_TRAILER and RUPT_F) else 300) - r.shape[0] - 30
            paste_rgba(frame, r, (OUT_W - r.shape[1]) // 2, y, min(1, (t - grp[0][0]) / 0.15))
            return

def frame_at(g):
    """Fotograma g aislado (para la revisión): busca el plano y el cuadro de origen."""
    pc = next(p for p in PIECES if p['f0'] <= g < p['f1']); i = g - pc['f0']
    if pc['kind'] == 'cine': return compose(g, pc, i, None, {})
    c = clips[pc['clip']]
    cap = cv2.VideoCapture(c['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, pc['src_f0'] + min(i, pc['n_talk'] - 1))
    ok, fr = cap.read(); cap.release()
    if not ok: fr = np.zeros((c['H'], c['W'], 3), np.uint8)
    return compose(g, pc, i, fr, {})

def timeline_text():
    mm = lambda f_: f"{int(f_ / FPS // 60)}:{f_ / FPS % 60:04.1f}"
    rows = [f"Duración {L:.0f}s (bloopers desde {MAIN_END / FPS:.0f}s) · subtítulos en altura {CAP_Y:.2f} y tamaño {CAP_SCALE:.2f} · música {VOLUMEN_MUSICA:.2f} · color {GRADE_MIX:.2f}"]
    rows += [f"tarjeta:{k} {mm(x['f0'])} «{x['text']}»" for k, x in enumerate(CARDS)]
    rows += [f"broll:{k} {mm(x['f0'])} {x['b']['name']} ({x['style']})" for k, x in enumerate(BROLLS)]
    rows += [f"titulo:{k} {mm(x['f0'])} «{x['text']}»" for k, x in enumerate(TITLES)]
    rows += [f"efecto:{k} {mm(x['f0'])} {x['kind']}" for k, x in enumerate(FX)]
    rows += [f"zoom:{k} {mm(x['f0'])}" for k, x in enumerate(ZOOMS)]
    return "\n".join(rows)

def review_sheet():
    """Hojas de contacto con los momentos importantes + tomas normales."""
    picks = [(int(0.8 * FPS), 'gancho')] + [((x['f0'] + x['f1']) // 2, f"trailer:{k}") for k, x in enumerate(TRAILER_TXT)]
    if RUPT_F: picks += [(RUPT_F + 10, 'ruptura')]
    for nm, lst in (('tarjeta', CARDS), ('broll', BROLLS), ('titulo', TITLES), ('efecto', FX), ('zoom', ZOOMS)):
        picks += [((x['f0'] + x['f1']) // 2, f"{nm}:{k}") for k, x in enumerate(lst)]
    picks += [(g_, 'normal') for g_ in np.linspace(FPS * 3, TOTAL_FRAMES - FPS * 4, 6).astype(int)]
    picks += [(CTA['f0'] + FPS, 'cierre')]
    picks = sorted({g_: lab for g_, lab in picks if 0 <= g_ < TOTAL_FRAMES}.items())[:24]
    tiles = []
    for g_, lab in picks:
        im = cv2.resize(frame_at(g_), (270, 480))
        cv2.rectangle(im, (0, 0), (270, 30), (0, 0, 0), -1)
        cv2.putText(im, f"{int(g_ / FPS // 60)}:{g_ / FPS % 60:04.1f} {lab}", (6, 21), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
        tiles.append(im)
    sheets = []
    for s_ in range(0, len(tiles), 8):
        row = tiles[s_:s_ + 8] + [np.zeros_like(tiles[0])] * (8 - len(tiles[s_:s_ + 8]))
        sheets.append(np.vstack([np.hstack(row[:4]), np.hstack(row[4:])]))
    return sheets

REVIEW_SCHEMA = {'type': 'object', 'additionalProperties': False,
                 'required': ['opinion', 'subtitulos_altura', 'subtitulos_tamano', 'volumen_musica', 'intensidad_color', 'quitar', 'cambiar_tarjetas'],
                 'properties': {'opinion': {'type': 'string'}, 'subtitulos_altura': {'type': 'number'},
                                'subtitulos_tamano': {'type': 'number'}, 'volumen_musica': {'type': 'number'},
                                'intensidad_color': {'type': 'number'},
                                'quitar': {'type': 'array', 'items': {'type': 'string'}},
                                'cambiar_tarjetas': {'type': 'array', 'items': {'type': 'object', 'additionalProperties': False,
                                                     'required': ['indice', 'texto'],
                                                     'properties': {'indice': {'type': 'integer'}, 'texto': {'type': 'string'}}}}}}
REVIEWS = []
if REVISION_IA and MOTOR_IA in ('local_qwen', 'claude'):
    for ronda in range(RONDAS_REVISION):
        sheets = review_sheet()
        for k, sh in enumerate(sheets): cv2.imwrite(f"{WORK}/revision_r{ronda + 1}_{k + 1}.jpg", sh, [cv2.IMWRITE_JPEG_QUALITY, 88])
        parts = [('img', base64.standard_b64encode(cv2.imencode('.jpg', sh, [cv2.IMWRITE_JPEG_QUALITY, 85])[1].tobytes()).decode()) for sh in sheets]
        parts.append(('text', f"""Eres el director creativo revisando el borrador de un TikTok vertical (promo de «{NOMBRE_SERVER}»): primero un TRÁILER
de cine (tomas lentas, barras negras, textos de tráiler, voz de narrador) que se rompe con un rayón de disco y pasa a HUMOR
(subtítulos amarillos, pantallas negras sarcásticas, b-roll, títulos 3D, zooms de risa) y termina en un cierre épico.
{TONOS.get(TONO, '')}
Arriba: fotogramas de muestra con su tiempo y tipo. Abajo: la línea de tiempo. Revisa como experto en retención:
- ¿Los subtítulos tapan la cara o quedan muy abajo (la interfaz de TikTok tapa el 20 % inferior)? Ajusta 'subtitulos_altura' (0.55-0.82) y 'subtitulos_tamano' (0.75-1.3).
- ¿El color se ve bien? 'intensidad_color' 0.4-1.2 (1 = como está).
- 'volumen_musica' 0.03-0.3 (no debe tapar la voz).
- 'quitar': elementos que sobran o estorban, con su etiqueta exacta (ej. "efecto:1", "tarjeta:0", "zoom:2"). Lista vacía si todo bien.
- 'cambiar_tarjetas': mejora el texto de pantallas negras flojas (máx. 7 palabras, en MAYÚSCULAS, con el tono indicado).
- 'opinion': 2-3 frases con tu crítica (en el tono indicado).
Línea de tiempo:
{timeline_text()}"""))
        try:
            rv = ask_llm(parts, REVIEW_SCHEMA, 'revision', max_tokens=4000)
        except Exception as e:
            print("⚠ La revisión con IA falló, sigo sin cambios:", str(e)[-800:]); break
        REVIEWS.append(rv)
        print(f"\n🎬 Ronda {ronda + 1} — {rv['opinion']}")
        CAP_Y = float(np.clip(rv['subtitulos_altura'], 0.55, 0.82)); CAP_SCALE = float(np.clip(rv['subtitulos_tamano'], 0.75, 1.3))
        GRADE_MIX = float(np.clip(rv['intensidad_color'], 0.4, 1.2))
        new_vol = float(np.clip(rv['volumen_musica'], 0.03, 0.3))
        for ch_ in rv['cambiar_tarjetas']:
            if 0 <= ch_['indice'] < len(CARDS) and ch_['texto'].strip():
                CARDS[ch_['indice']]['text'] = ch_['texto'].strip().upper()[:60]
        drop = {}
        for tag in rv['quitar']:
            m_ = re.match(r'(tarjeta|broll|titulo|efecto|zoom):(\d+)$', tag.strip())
            if m_: drop.setdefault(m_.group(1), set()).add(int(m_.group(2)))
        lists = {'tarjeta': 'CARDS', 'broll': 'BROLLS', 'titulo': 'TITLES', 'efecto': 'FX', 'zoom': 'ZOOMS'}
        for nm, var in lists.items():
            if nm in drop: globals()[var] = [x for k, x in enumerate(globals()[var]) if k not in drop[nm]]
        _cap_cache.clear(); _card_cache.clear()
        print(f"   subtítulos altura {CAP_Y:.2f} tamaño {CAP_SCALE:.2f} · color {GRADE_MIX:.2f} · música {new_vol:.2f} · "
              f"quitados: {', '.join(rv['quitar']) or 'nada'} · tarjetas cambiadas: {len(rv['cambiar_tarjetas'])}")
        if abs(new_vol - VOLUMEN_MUSICA) > 1e-3:
            VOLUMEN_MUSICA = new_vol; build_audio(VOLUMEN_MUSICA)
    final_sheet = review_sheet()
    for k, sh in enumerate(final_sheet): cv2.imwrite(f"{WORK}/revision_final_{k + 1}.jpg", sh, [cv2.IMWRITE_JPEG_QUALITY, 88])
    if IN_COLAB:
        from IPython.display import Image as _Im, display
        display(_Im(f"{WORK}/revision_final_1.jpg", width=900))
else:
    print("Revisión con IA desactivada.")
llm_stop()                                                   # libera la GPU para el render

In [ ]:
#@title 9 · Render final 9:16 (1080×1920) para TikTok / Reels / Shorts
OUT = os.path.join(WORK, f"{re.sub(r'[^A-Za-z0-9]+', '_', NOMBRE_SERVER).strip('_').lower()}_promo_tiktok.mp4")
cmd = ['ffmpeg', '-y', '-v', 'error', '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', f'{OUT_W}x{OUT_H}', '-r', str(FPS),
       '-i', '-', '-i', MIX_WAV, '-map', '0:v', '-map', '1:a', *VENC, '-pix_fmt', 'yuv420p',
       '-color_primaries', 'bt709', '-color_trc', 'bt709', '-colorspace', 'bt709',
       '-c:a', 'aac', '-b:a', '256k', '-t', f"{L:.4f}", '-movflags', '+faststart', OUT]
pipe = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE, cwd=WORK)
t0 = time.time(); g = 0; ST = {}
for pc in PIECES:
    cap = None
    if pc['kind'] == 'talk':
        c = clips[pc['clip']]; cap = cv2.VideoCapture(c['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, pc['src_f0'])
    last = None
    for i in range(pc['n']):
        fr = None
        if cap is not None:
            ok, fr = cap.read()
            fr = fr if ok else (last if last is not None else np.zeros((c['H'], c['W'], 3), np.uint8)); last = fr
        f = compose(g, pc, i, fr, ST)
        try:
            pipe.stdin.write(f.tobytes())
        except BrokenPipeError:
            raise RuntimeError("ffmpeg se detuvo: " + pipe.stderr.read().decode()[-2000:])
        g += 1
    if cap is not None: cap.release()
    print(f"\r  {g}/{TOTAL_FRAMES} frames ({g / max(1e-6, time.time() - t0):.1f} fps)", end='')
pipe.stdin.close(); err = pipe.stderr.read().decode(); pipe.wait()
if pipe.returncode != 0:
    raise RuntimeError(err[-2000:])
print(f"\n✓ {OUT}")

In [ ]:
#@title 10 · Guardar en Drive (carpeta Edicion_IA) + guion, notas y revisión de la IA
notes = [f"# {NOMBRE_SERVER} — promo TikTok", "", PLAN.get('resumen', ''), "",
         f"- Duración: {L:.1f}s · {len(PIECES)} planos · gancho: **{PLAN['gancho']}** · CTA: **{PLAN['cta']}**", "",
         "| # | Clip | Lo que dices | En grande | Pantalla negra | Capítulo 3D | B-roll | Efecto | Canción |",
         "|---|---|---|---|---|---|---|---|---|"]
for k, g_ in enumerate(PLAN['guion']):
    if g_['tipo'] == 'toma':
        notes.append(f"| {k + 1} | 🎞 {g_['archivo']}{' (cámara lenta)' if g_['camara_lenta'] else ''} | *{g_['texto_trailer']}* | — | — | — | — | tráiler | — |")
        continue
    p = byid[g_['id']]
    song = f"{g_['pedido_cancion']} ({g_['segundos_cancion']:.0f}s{', ' + g_['cancion'] if g_['cancion'] else ', en TikTok'})" if g_['pedido_cancion'] else '—'
    br = f"{g_['broll']} ({g_['estilo_broll']})" if g_['broll'] else '—'
    notes.append(f"| {k + 1} | {clips[p['clip']]['name']} @ {p['start']:.1f}s | {p['text']} | {g_['enfasis'] or '—'} | "
                 f"{g_['tarjeta'] or '—'} | {g_['capitulo'] or '—'} | {br} | {g_['efecto'] or '—'} | {song} |")
if REVIEWS:
    notes += ["", "## Revisión de la IA"] + [f"- Ronda {k + 1}: {r_['opinion']}" for k, r_ in enumerate(REVIEWS)]
if CREDITO_MUSICA.strip():
    notes += ["", f"**Créditos:** {CREDITO_MUSICA.strip()}"]
open(f"{WORK}/guion_promo_tiktok.md", 'w', encoding='utf-8').write("\n".join(notes))

# Lista para poner las canciones desde "Agregar sonido" en TikTok
mmss = lambda x: f"{int(x // 60)}:{int(x % 60):02d}"
lst = [f"Canciones para poner en TikTok — {NOMBRE_SERVER}", "",
       "En TikTok: Agregar sonido → busca la canción → recórtala → colócala en el minuto indicado.", ""]
for e in sorted(SONG_MARKS + SONG_EVENTS, key=lambda e: e['t0']):
    where = "ya va dentro del video" if e['file'] else "ponla desde TikTok"
    lst.append(f"{mmss(e['t0'])}  →  «{e['name']}»  ({e['dur']:.0f} s, {where})")
if len(lst) == 4: lst.append("(no se detectaron pedidos de canción en el video)")
open(f"{WORK}/canciones_para_tiktok.txt", 'w', encoding='utf-8').write("\n".join(lst) + "\n")
print("\n".join(lst[4:]))
for p in [OUT, f"{WORK}/guion_promo_tiktok.md", f"{WORK}/transcripcion.txt", f"{WORK}/canciones_para_tiktok.txt"] + sorted(glob.glob(f"{WORK}/revision_*.jpg")):
    shutil.copy(p, OUT_DIR); print("→", os.path.join(OUT_DIR, os.path.basename(p)))
if IN_COLAB:
    from IPython.display import Video, display
    prev = f"{WORK}/preview.mp4"
    run(['ffmpeg', '-y', '-v', 'error', '-i', OUT, '-vf', 'scale=360:-2', '-c:v', 'libx264', '-crf', '30', '-c:a', 'aac', '-b:a', '96k', prev])
    display(Video(prev, embed=True, width=300))
print("\n¿Ajustes? Edita PLAN['guion'] ('enfasis', 'tarjeta', 'broll', 'estilo_broll', 'capitulo', 'efecto'), PLAN['gancho'] / PLAN['cta'] y vuelve a correr las celdas 6–10.")